# Coffee Leaf Disease Classification (DINOv2 ViT-L) — Source Code

**Author:** Kambale Muhesi Muyisa


# Coffee Leaf Disease Classification — **Pipeline v2.1**
## DINOv2 ViT-L + K-fold Ensemble + Multi-scale TTA

| Parameter | v1 | **v2** |
|---|---|---|
| Backbone | DINOv2 **ViT-B/14** (86M, 768d) | **DINOv2 ViT-L/14** (304M, **1024d**) |
| Handcrafted | HSV+GLCM+LBP+Gabor+ColorMom = 171d | **HSV+Gabor = 136d** (drop redundant) |
| Fusion α | 0.9 (CV-tuned) | **0.95** (deep heavily favored) |
| Fine-tune | 1 model, n_unfreeze=3 | **5-fold ensemble**, n_unfreeze=3 |
| Augment | Mixup α=0.2 | Mixup **+ CutMix** (random switch) |
| TTA | 4 flips @ 224 | **3 scale × 4 flip = 12 passes** |
| Aggregation | Softmax 1 model | **Average softmax 5 model** |
| Datasets | DS1+...+DS5 stratified 70/15/15 | unchanged |

> **Expectation:** v1 (Test F1 ~95–98% after merging datasets) → v2 adds **+1.5–3pp F1**
> mainly from ViT-L (+1–2pp) and the 5-fold ensemble (+1–2pp). Dropping redundant HC streamlines the pipeline.


## v2 improvements — technical details

| # | Item | Improvement |
|---|---|---|
| 1 | **Backbone** | ViT-L/14 replaces ViT-B/14: 4× params, 1024-dim CLS instead of 768. Same DINOv2 self-supervised paradigm. |
| 2 | **Trimmed HC** | Drop **LBP (10d)** — uniform LBP is nearly redundant with CNN texture. Drop **GLCM (16d)** — DINOv2 already encodes statistical texture. Drop **Color Moments (9d)** — duplicates the HSV histogram. **Keep HSV (96d)** + **Gabor (40d)** = 136d. |
| 3 | **K-fold ensemble FT** | 5 FT models trained on 5 different folds (StratifiedKFold on train+val combined). Evaluated with `mean(softmax_k for k in 1..5).argmax()` — substantially reduces variance. |
| 4 | **CutMix** | Alongside Mixup: 50% of batches use Mixup, 50% use CutMix (rectangle-paste). Two complementary regularizations — the original paper reports +0.3pp F1. |
| 5 | **Multi-scale TTA** | Forward at 3 scales (224, 256, 288) × 4 flips = **12 passes** per image. Average softmax. |
| 6 | **EMA + cosine LR** | Kept from v1 — smooth convergence. |
| 7 | **Save K-fold weights** | Save 5 FT state_dicts + 1 ensemble metadata for reloading. |


## 0. Install libraries

In [ ]:
!pip install -q transformers>=4.56.0 scikit-image xgboost lightgbm joblib

## 1. Imports & Configuration (ViT-L)

In [ ]:
# ─── Mount Google Drive (Google Colab) ─────────────────────
# Datasets and outputs live in your Google Drive.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Not running on Colab (or Drive already mounted):', e)

import os, random, time, warnings, copy, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from collections import Counter
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T

from transformers import AutoImageProcessor, AutoModel

import cv2
from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score, recall_score,
)
from joblib import Parallel, delayed
from scipy import stats as sps  # McNemar / bootstrap

# ---- SEED -----------------------------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Deterministic DataLoader workers (fix v2.0: workers were not seeded)
def seed_worker(worker_id):
    s = (torch.initial_seed() + worker_id) % (2**32)
    np.random.seed(s); random.seed(s)
GENERATOR = torch.Generator(); GENERATOR.manual_seed(SEED)

# ---- DEVICE ---------------------------------------------
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# ---- 4 TARGET CLASSES ----------------------------------
TARGET_CLASSES = ['Healthy', 'Miner', 'Phoma', 'Rust']
CLASS_VI = {
    'Healthy': 'Healthy',
    'Miner'  : 'Leaf Miner',
    'Phoma'  : 'Phoma',
    'Rust'   : 'Rust',
}
COLORS = ['#4CAF50', '#FF9800', '#795548', '#F44336']

# ---- BACKBONE: DINOv2 ViT-L/14 (was ViT-B/14 in v1) -------------
DINOV2_MODEL = 'facebook/dinov2-large'
DEEP_DIM     = 1024
DINOV2_BASE  = 'facebook/dinov2-base'   # used by ViT-B ablation
DEEP_DIM_B   = 768

# ---- CONFIG ---------------------------------------------
IMG_SIZE       = 224
EXTRACT_BATCH  = 32       # frozen extraction (no gradient)
FT_BATCH_SIZE  = 16       # fine-tune (reduced from 32 because ViT-L is larger)
NUM_WORKERS    = 4
N_FOLDS        = 5        # K-fold ensemble
FT_EPOCHS      = 18       # per-fold (5 folds × 18 epochs ≈ 2-2.5h on T4)
PATIENCE       = 4

# ---- ABLATION FLAGS (toggle to cut compute) -------------
RUN_ABLATION_NO_HC      = True    # ~ 1 min  (re-fit ELM with alpha=1.0)
RUN_ABLATION_VITB       = True    # ~ 5 min  (extract + ELM with ViT-B)
RUN_ABLATION_SINGLE_FT  = True    # ~25 min  (1 FT model on (train+val) split)
RUN_TEMPERATURE_SCALING = True    # ~30 sec  (re-extract val probs)
N_BOOTSTRAP             = 1000    # bootstrap iterations for 95% CI

print(f'Backbone : {DINOV2_MODEL}  (CLS dim = {DEEP_DIM})')
print(f'K-fold   : {N_FOLDS}, epochs/fold = {FT_EPOCHS}')
print(f'Ablation : NO_HC={RUN_ABLATION_NO_HC}  ViT-B={RUN_ABLATION_VITB}  '
      f'SINGLE_FT={RUN_ABLATION_SINGLE_FT}  T-scaling={RUN_TEMPERATURE_SCALING}')
print(f'Bootstrap: {N_BOOTSTRAP} resamples for 95% CI')
print('Imports & configuration done!')

# ─── Output directory (Google Drive) ─────────────────────
# All figures, CSVs and saved models go here. Edit if you prefer another folder.
OUTPUT_DIR = Path('/content/drive/MyDrive/coffee_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Gather data + deduplicate + stratified split

| Step | Description |
|---|---|
| 1 | Scan 5 datasets (DS1–DS5) |
| 2 | Merge all → dedup MD5 + dHash 17×16 (256-bit) |
| 3 | Stratified split 70/15/15 (train/val/test) |


In [ ]:
# PATH CONFIGURATION  ─────────────────────────────────────
import hashlib
from io import BytesIO

# Datasets are in your Google Drive. Edit DRIVE_ROOT; keep each sub-folder name
# in sync with how the dataset is actually stored in your Drive.
DRIVE_ROOT = Path('/content/drive/MyDrive/coffee_datasets')

DS1_TRAIN = DRIVE_ROOT / 'coffee-leaf-disease-dataset' / 'dataset' / 'Train'
DS1_MAP = {'Healthy': 'Healthy', 'Miner': 'Miner', 'Phoma': 'Phoma', 'Rust': 'Rust'}

DS2_ROOT = DRIVE_ROOT / 'coffee-leaves-disease' / 'Coffee Leave Disease'
DS2_MAP  = {'Healty': 'Healthy', 'Rust': 'Rust'}

DS3_ROOT = DRIVE_ROOT / 'coffee-dataset-mendeley' / 'coffee dataset'
DS3_MAP  = {'Health leaves': 'Healthy', 'leaf rust': 'Rust', 'phoma': 'Phoma'}

DS4_ROOT = DRIVE_ROOT / 'rocole-a-robusta-coffee-leaf-images-dataset'
DS4_MAP  = {'coffee___healthy': 'Healthy', 'coffee___rust': 'Rust'}

DS5_ROOT = DRIVE_ROOT / 'jmuben-coffee-dataset' / 'JMuBEN'
DS5_MAP  = {'Healthy': 'Healthy', 'Miner': 'Miner', 'Phoma': 'Phoma', 'Leaf rust': 'Rust'}

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}


def scan_folder(root, class_map, source_tag='', verbose=True):
    samples = []
    if not root.exists():
        print(f'  [!] Not found: {root}')
        return samples
    if verbose:
        print(f'\n  [{source_tag}]  {root}')
    sub_total = 0
    for folder in sorted(root.iterdir()):
        if not folder.is_dir():
            continue
        cls_std = class_map.get(folder.name)
        n_imgs = sum(1 for p in folder.iterdir() if p.suffix.lower() in IMG_EXTS)
        if cls_std is None or cls_std not in TARGET_CLASSES:
            if verbose:
                print(f'      - {folder.name:<25} : {n_imgs:>6}  [SKIP]')
            continue
        imgs = sorted(p for p in folder.iterdir() if p.suffix.lower() in IMG_EXTS)
        samples.extend((p, cls_std, source_tag) for p in imgs)
        sub_total += len(imgs)
        if verbose:
            print(f'      - {folder.name:<25} -> {cls_std:<8} : {len(imgs):>6}')
    if verbose:
        print(f'      {"KEPT":<25}            : {sub_total:>6}')
    return samples


def hash_image(path):
    """MD5 (exact) + dHash 17x16 = 256-bit (perceptual)."""
    try:
        with open(path, 'rb') as f:
            data = f.read()
        md5 = hashlib.md5(data).hexdigest()
        img = Image.open(BytesIO(data)).convert('L').resize((17, 16), Image.LANCZOS)
        arr = np.asarray(img, dtype=np.int16)
        dh = (arr[:, 1:] > arr[:, :-1]).tobytes()
        return md5, dh
    except Exception:
        return None, None


def dedupe(samples_with_source, ref_md5=None, ref_dhash=None, desc='Dedup'):
    seen_md5, seen_dh = set(), set()
    ref_md5 = ref_md5 or set()
    ref_dh  = ref_dhash or set()
    kept = []
    total = dict(total=0, leak=0, dup_exact=0, dup_perceptual=0, error=0, kept=0)
    per_src = {}
    for path, label, src in tqdm(samples_with_source, desc=desc):
        total['total'] += 1
        per_src.setdefault(src, dict(total=0, leak=0, dup_exact=0,
                                     dup_perceptual=0, error=0, kept=0))
        per_src[src]['total'] += 1
        m, d = hash_image(path)
        if m is None or d is None:
            total['error'] += 1; per_src[src]['error'] += 1; continue
        if m in ref_md5 or d in ref_dh:
            total['leak'] += 1; per_src[src]['leak'] += 1; continue
        if m in seen_md5:
            total['dup_exact'] += 1; per_src[src]['dup_exact'] += 1; continue
        if d in seen_dh:
            total['dup_perceptual'] += 1; per_src[src]['dup_perceptual'] += 1; continue
        seen_md5.add(m); seen_dh.add(d)
        kept.append((path, label))
        per_src[src]['kept'] += 1
        total['kept'] += 1
    return kept, total, per_src, seen_md5, seen_dh


# ---- Scan 5 datasets -------------------------------------
print('=' * 70)
print('  SCAN DATASETS')
print('=' * 70)
ds1 = scan_folder(DS1_TRAIN, DS1_MAP, 'DS1')
ds2 = scan_folder(DS2_ROOT,  DS2_MAP, 'DS2')
ds3 = scan_folder(DS3_ROOT,  DS3_MAP, 'DS3')
ds4 = scan_folder(DS4_ROOT,  DS4_MAP, 'DS4')
ds5 = scan_folder(DS5_ROOT,  DS5_MAP, 'DS5')

all_raw = ds1 + ds2 + ds3 + ds4 + ds5
print('\n' + '=' * 50)
print('  SUMMARY (before dedup)')
print('=' * 50)
for name, s in [('DS1', ds1), ('DS2', ds2), ('DS3', ds3),
                ('DS4', ds4), ('DS5 (JMuBEN)', ds5)]:
    print(f'  {name:<20}: {len(s):>7,}')
print('-' * 50)
print(f'  {"TOTAL":<20}: {len(all_raw):>7,}')


In [ ]:
# DEDUPE COMBINED + STRATIFIED SPLIT  ─────────────────────
print('\nDedup combined (MD5 + dHash 17x16):')
all_kept, all_stats, all_per, _, _ = dedupe(all_raw, desc='Dedup ALL')

print('\n' + '=' * 82)
print('  DEDUP DETAIL BY SOURCE')
print('=' * 82)
print(f'{"Source":<14} {"Raw":>8} {"Exact":>8} {"Percep":>8} {"Err":>6} {"Kept":>8}')
print('-' * 82)
for src, s in all_per.items():
    print(f'{src:<14} {s["total"]:>8} {s["dup_exact"]:>8} '
          f'{s["dup_perceptual"]:>8} {s["error"]:>6} {s["kept"]:>8}')
print('-' * 82)
print(f'{"TOTAL":<14} {all_stats["total"]:>8} {all_stats["dup_exact"]:>8} '
      f'{all_stats["dup_perceptual"]:>8} {all_stats["error"]:>6} {all_stats["kept"]:>8}')

removed = all_stats['total'] - all_stats['kept']
pct = removed / max(all_stats['total'], 1) * 100
print(f'\n==> Removed {removed:,} / {all_stats["total"]:,} images ({pct:.1f}%)')


# ---- Stratified split: 70/15/15 ─────────────────────────
paths_all  = [p for p, _ in all_kept]
labels_all = [l for _, l in all_kept]

p_tv, p_te, l_tv, l_te = train_test_split(
    paths_all, labels_all, test_size=0.15,
    stratify=labels_all, random_state=SEED,
)
p_tr, p_va, l_tr, l_va = train_test_split(
    p_tv, l_tv, test_size=0.15 / 0.85,
    stratify=l_tv, random_state=SEED,
)

train_samples = list(zip(p_tr, l_tr))
val_samples   = list(zip(p_va, l_va))
test_samples  = list(zip(p_te, l_te))


def class_counts(samples):
    c = Counter(l for _, l in samples)
    return {cls: c.get(cls, 0) for cls in TARGET_CLASSES}


print('\n' + '=' * 56)
print('  CLASS DISTRIBUTION')
print('=' * 56)
print(f'{"Class":<12} {"Train":>10} {"Val":>10} {"Test":>10}')
print('-' * 56)
c_tr, c_va, c_te = class_counts(train_samples), class_counts(val_samples), class_counts(test_samples)
for cls in TARGET_CLASSES:
    print(f'{cls:<12} {c_tr[cls]:>10,} {c_va[cls]:>10,} {c_te[cls]:>10,}')
print('-' * 56)
print(f'{"TOTAL":<12} {len(train_samples):>10,} {len(val_samples):>10,} {len(test_samples):>10,}')


## 2.1 Dedup audit — why is 89.2% of the data removed?

A reviewer will ask: is the `dHash 17×16` dedup (272-bit, exact match) too loose or too strict?
The analysis below explains:
1. **Histogram of Hamming distance** between pairs with different MD5 — if the distribution skews toward 0, JMuBEN really does contain many near-duplicate (perceptual duplicate) images.
2. **6 representative duplicate pairs** from JMuBEN — visual sanity check.
3. **Top-10 largest clusters** (by dHash) — groups of near-identical images.

> If the Hamming distribution clusters around 0 on JMuBEN, other papers reporting high accuracy on this dataset may be measuring on redundant data.


In [ ]:
# DEDUP AUDIT  ─────────────────────────────────────────
# Compute dHash 17x16 for ALL raw images (for illustration). For large datasets
# you can subsample if needed.
import random as _rnd

DEDUP_AUDIT_SAMPLES = 6000   # cap the number of images analyzed if raw is too large
AUDIT_SOURCE_FILTER = 'DS5'  # JMuBEN — where dedup removes 97%

raw_audit = [s for s in all_raw if s[2] == AUDIT_SOURCE_FILTER]
if len(raw_audit) > DEDUP_AUDIT_SAMPLES:
    _rnd.seed(SEED)
    raw_audit = _rnd.sample(raw_audit, DEDUP_AUDIT_SAMPLES)
print(f'Audit on {AUDIT_SOURCE_FILTER}: {len(raw_audit):,} images')

# Hash all (path, label) and store the dHash bytes
def _bits_from_dh(dh_bytes):
    return np.unpackbits(np.frombuffer(dh_bytes, dtype=np.uint8))

audit_paths, audit_bits = [], []
for path, _, _ in tqdm(raw_audit, desc='Audit hash'):
    _, dh = hash_image(path)
    if dh is None: continue
    audit_paths.append(path)
    audit_bits.append(_bits_from_dh(dh))
audit_bits = np.stack(audit_bits, axis=0).astype(np.uint8)
print(f'Hash matrix: {audit_bits.shape}  (n_imgs, 272 bits)')

# Sample random pairs and compute Hamming distance distribution
N_PAIRS = 200_000
_rnd.seed(SEED)
n = audit_bits.shape[0]
i_a = np.random.randint(0, n, size=N_PAIRS)
i_b = np.random.randint(0, n, size=N_PAIRS)
ok  = i_a != i_b
i_a, i_b = i_a[ok], i_b[ok]
ham = np.count_nonzero(audit_bits[i_a] != audit_bits[i_b], axis=1)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(ham, bins=80, color='#1f77b4', alpha=0.8, edgecolor='black', linewidth=0.3)
ax.axvline(0, color='red', ls='--', label='exact match (current threshold)')
for thr in [5, 10, 20]:
    ax.axvline(thr, color='orange', ls=':', alpha=0.6,
               label=f'<= {thr} bits' if thr == 5 else None)
ax.set_xlabel('Hamming distance (out of 272 bits)')
ax.set_ylabel('Pair count')
ax.set_title(f'Pair-wise Hamming distance on {AUDIT_SOURCE_FILTER} '
             f'({len(ham):,} random pairs)', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/coffee_outputs/fig_dedup_hamming.png', dpi=160, bbox_inches='tight')
plt.show()

# Cluster images by exact dHash (computed over ALL audit_paths, not the subsampled pairs)
from collections import defaultdict
cluster = defaultdict(list)
for p, bits in zip(audit_paths, audit_bits):
    key = bits.tobytes()
    cluster[key].append(p)
sizes = sorted([(len(v), k) for k, v in cluster.items()], reverse=True)
print(f'\nTop 10 largest clusters (images sharing the same dHash):')
for sz, k in sizes[:10]:
    print(f'  cluster size = {sz}')

# Visualise 6 representative duplicate pairs (Hamming = 0)
n_show = min(6, sum(1 for sz, _ in sizes if sz >= 2))
shown = 0
fig, axes = plt.subplots(n_show, 2, figsize=(6, 2.5 * n_show))
if n_show == 1: axes = axes.reshape(1, 2)
for sz, k in sizes:
    if sz < 2 or shown >= n_show: break
    p1, p2 = cluster[k][0], cluster[k][1]
    for j, p in enumerate([p1, p2]):
        try:
            img = Image.open(p).convert('RGB').resize((180, 180))
            axes[shown, j].imshow(img)
        except Exception: pass
        axes[shown, j].axis('off')
        axes[shown, j].set_title(f'cluster size={sz}\n{p.parent.name}/{p.name[:18]}',
                                 fontsize=8)
    shown += 1
fig.suptitle(f'6 representative duplicate pairs ({AUDIT_SOURCE_FILTER}, dHash exact match)',
             fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/coffee_outputs/fig_dedup_pairs.png', dpi=160, bbox_inches='tight')
plt.show()

# Summary
n0    = int((ham == 0).sum())
n_le5 = int((ham <= 5).sum())
print(f'\nAudit summary ({AUDIT_SOURCE_FILTER}):')
print(f'  Hamming == 0   (current dedup) : {n0:>7,}  ({n0/len(ham)*100:.2f}%)')
print(f'  Hamming <= 5   (stricter)      : {n_le5:>7,}  ({n_le5/len(ham)*100:.2f}%)')
print(f'  Median Hamming = {int(np.median(ham))} bits, P95 = {int(np.percentile(ham,95))}')
print(f'  -> If Hamming<=5 is a more reasonable threshold, conclusion: {AUDIT_SOURCE_FILTER} really has many internal near-duplicates.')


## 2.2 EDA — Data visualization charts

After dedup + stratified split, this is the visual analysis to better understand the data and to defend it in the report. The charts cover:

| # | Chart | Meaning |
|---|---|---|
| 1 | Grouped bar: class x split | Check class distribution across train/val/test, spot imbalance between splits |
| 2 | Overall pie | Overall class proportions (after dedup) — is there class imbalance? |
| 3 | Bar raw vs kept by source | Which dataset loses the most images during dedup? |
| 4 | Scatter width x height | Image size distribution — does resizing to 224 lose information? |
| 5 | Bar % class | Verify stratified split: class % is the SAME across the 3 splits |
| 6 | Sample image grid | Visual inspection of the classes (to catch label noise / outliers) |


In [ ]:
# EDA: DATA DISTRIBUTION ANALYSIS AFTER SPLIT  ─────────────
# 5 charts in 1 figure:
#   (1) Class distribution per split (train/val/test)
#   (2) Pie chart of overall class distribution (after dedup)
#   (3) Contribution of each source dataset (DS1..DS5) before and after dedup
#   (4) Image size histogram (sample) - width vs height
#   (5) 4x3 grid of sample images per class

import matplotlib.image as mpimg
from PIL import Image as PILImage
import random as _rnd_eda

# ---- (1) Class distribution per split  -------------------
fig = plt.figure(figsize=(20, 12))
gs = fig.add_gridspec(3, 4, hspace=0.42, wspace=0.32)

ax1 = fig.add_subplot(gs[0, 0:2])
splits = ['Train', 'Val', 'Test']
split_counts = [c_tr, c_va, c_te]
x_pos = np.arange(len(TARGET_CLASSES))
w_bar = 0.26
for i, (sp, cd) in enumerate(zip(splits, split_counts)):
    vals = [cd[c] for c in TARGET_CLASSES]
    bars = ax1.bar(x_pos + (i - 1) * w_bar, vals, w_bar, label=sp,
                   color=COLORS, alpha=0.55 + i * 0.22, edgecolor='black', linewidth=0.6)
    for b, v in zip(bars, vals):
        ax1.text(b.get_x() + b.get_width() / 2, b.get_height() + max(vals) * 0.01,
                 f'{v}', ha='center', fontsize=8, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels([CLASS_VI[c] for c in TARGET_CLASSES], fontsize=10)
ax1.set_ylabel('Number of images', fontweight='bold')
ax1.set_title('(1) Class distribution per split (after dedup + stratified)', fontweight='bold', fontsize=12)
ax1.legend(loc='upper right', fontsize=9)
ax1.grid(axis='y', alpha=0.3)

# ---- (2) Pie chart total class distribution --------------
ax2 = fig.add_subplot(gs[0, 2])
total_per_cls = {c: c_tr[c] + c_va[c] + c_te[c] for c in TARGET_CLASSES}
sizes = [total_per_cls[c] for c in TARGET_CLASSES]
labels_pie = [f'{CLASS_VI[c]}\n{v:,} ({v/sum(sizes)*100:.1f}%)' for c, v in zip(TARGET_CLASSES, sizes)]
wedges, _ = ax2.pie(sizes, labels=labels_pie, colors=COLORS, startangle=90,
                    wedgeprops=dict(edgecolor='white', linewidth=2),
                    textprops=dict(fontsize=9, fontweight='bold'))
ax2.set_title(f'(2) Overall class distribution (n={sum(sizes):,})', fontweight='bold', fontsize=12)

# ---- (3) Source dataset contribution ---------------------
ax3 = fig.add_subplot(gs[0, 3])
src_names = list(all_per.keys())
raw_counts = [all_per[s]['total'] for s in src_names]
kept_counts = [all_per[s]['kept'] for s in src_names]
xs = np.arange(len(src_names))
ax3.bar(xs - 0.2, raw_counts, 0.4, label='Raw', color='#90A4AE', alpha=0.85)
ax3.bar(xs + 0.2, kept_counts, 0.4, label='Kept (after dedup)', color='#2196F3', alpha=0.85)
for i, (r, k) in enumerate(zip(raw_counts, kept_counts)):
    ax3.text(i - 0.2, r + max(raw_counts) * 0.01, f'{r}', ha='center', fontsize=7)
    ax3.text(i + 0.2, k + max(raw_counts) * 0.01, f'{k}', ha='center', fontsize=7, fontweight='bold')
ax3.set_xticks(xs)
ax3.set_xticklabels(src_names, fontsize=9)
ax3.set_ylabel('Number of images', fontweight='bold')
ax3.set_title('(3) Contribution per dataset (before/after dedup)', fontweight='bold', fontsize=12)
ax3.legend(fontsize=8)
ax3.grid(axis='y', alpha=0.3)

# ---- (4) Image size distribution (sampled) ---------------
ax4 = fig.add_subplot(gs[1, 0:2])
SAMPLE_DIM = 600
sample_paths = _rnd_eda.Random(SEED).sample(
    [p for p, _ in all_kept], min(SAMPLE_DIM, len(all_kept)))
widths, heights = [], []
for p in sample_paths:
    try:
        with PILImage.open(p) as im:
            widths.append(im.width)
            heights.append(im.height)
    except Exception:
        continue
ax4.scatter(widths, heights, s=14, alpha=0.45, c='#1976D2', edgecolor='white', linewidth=0.3)
ax4.axhline(y=224, color='red', linestyle='--', linewidth=1, alpha=0.6, label='IMG_SIZE=224')
ax4.axvline(x=224, color='red', linestyle='--', linewidth=1, alpha=0.6)
ax4.set_xlabel('Width (px)', fontweight='bold')
ax4.set_ylabel('Height (px)', fontweight='bold')
ax4.set_title(f'(4) Image sizes (n={len(widths)} random samples)', fontweight='bold', fontsize=12)
ax4.legend(loc='upper right', fontsize=9)
ax4.grid(alpha=0.3)
ax4.text(0.02, 0.97,
         f'Width:  min={min(widths)}  max={max(widths)}  median={int(np.median(widths))}\n'
         f'Height: min={min(heights)} max={max(heights)} median={int(np.median(heights))}',
         transform=ax4.transAxes, fontsize=9, va='top',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.85, edgecolor='gray'))

# ---- (5) Class balance ratio per split -------------------
ax5 = fig.add_subplot(gs[1, 2:4])
for i, (sp, cd) in enumerate(zip(splits, split_counts)):
    total = sum(cd.values())
    pcts = [cd[c] / total * 100 for c in TARGET_CLASSES]
    bars = ax5.bar(x_pos + (i - 1) * w_bar, pcts, w_bar, label=sp,
                   color=COLORS, alpha=0.55 + i * 0.22, edgecolor='black', linewidth=0.6)
    for b, v in zip(bars, pcts):
        ax5.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.5,
                 f'{v:.1f}%', ha='center', fontsize=8, fontweight='bold')
ax5.set_xticks(x_pos)
ax5.set_xticklabels([CLASS_VI[c] for c in TARGET_CLASSES], fontsize=10)
ax5.set_ylabel('Proportion (%)', fontweight='bold')
ax5.set_title('(5) Class proportion per split (stratified check)', fontweight='bold', fontsize=12)
ax5.legend(loc='upper right', fontsize=9)
ax5.grid(axis='y', alpha=0.3)

# ---- (6) Sample images grid (4 classes x 3 samples) ------
samples_by_cls = {c: [] for c in TARGET_CLASSES}
for p, l in all_kept:
    if len(samples_by_cls[l]) < 3:
        samples_by_cls[l].append(p)
    if all(len(samples_by_cls[c]) >= 3 for c in TARGET_CLASSES):
        break

ax_grid = fig.add_subplot(gs[2, :])
ax_grid.axis('off')
ax_grid.set_title('(6) Representative sample images per class (3 images/class)',
                  fontweight='bold', fontsize=12, pad=10)
inner_gs = gs[2, :].subgridspec(1, 12, wspace=0.08)
for ci, cls in enumerate(TARGET_CLASSES):
    for si, ipath in enumerate(samples_by_cls[cls]):
        sub_ax = fig.add_subplot(inner_gs[0, ci * 3 + si])
        try:
            img = PILImage.open(ipath).convert('RGB')
            img.thumbnail((400, 400), PILImage.LANCZOS)
            sub_ax.imshow(img)
        except Exception:
            sub_ax.text(0.5, 0.5, 'load err', ha='center', va='center')
        sub_ax.set_xticks([]); sub_ax.set_yticks([])
        for s in sub_ax.spines.values():
            s.set_edgecolor(COLORS[ci]); s.set_linewidth(2.5)
        if si == 1:
            sub_ax.set_title(CLASS_VI[cls], fontsize=10, fontweight='bold',
                             color=COLORS[ci], pad=4)

fig.suptitle('EDA — Data distribution analysis (post-dedup, stratified 70/15/15)',
             fontsize=14, fontweight='bold', y=0.995)
plt.savefig('/content/drive/MyDrive/coffee_outputs/fig_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n[OK] EDA chart saved -> /content/drive/MyDrive/coffee_outputs/fig_eda.png')


## 3. Extract CLS features — DINOv2 ViT-L (1024-dim)

In [ ]:
# Load DINOv2 ViT-L  ─────────────────────────────────────
print(f'Loading {DINOV2_MODEL}...')
dinov2_processor = AutoImageProcessor.from_pretrained(DINOV2_MODEL)
backbone = AutoModel.from_pretrained(DINOV2_MODEL).to(DEVICE).eval()
for p in backbone.parameters():
    p.requires_grad = False
print(f'OK — output {DEEP_DIM}-dim (CLS token)')


class CoffeeDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        return img, label, str(path)


def dinov2_collate(batch):
    imgs, labels, paths = zip(*batch)
    inputs = dinov2_processor(
        images=list(imgs), return_tensors='pt',
        size={'height': IMG_SIZE, 'width': IMG_SIZE},
    )
    return inputs, list(labels), list(paths)


@torch.no_grad()
def extract_deep_features(samples, desc='Extract DINOv2'):
    ds = CoffeeDataset(samples)
    loader = DataLoader(
        ds, batch_size=EXTRACT_BATCH, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'),
        collate_fn=dinov2_collate,
    )
    feats, labels, paths = [], [], []
    use_amp = (DEVICE.type == 'cuda')
    for inputs, lbls, pth in tqdm(loader, desc=desc):
        pixel_values = inputs['pixel_values'].to(DEVICE, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=use_amp):
            out = backbone(pixel_values=pixel_values)
            cls = out.last_hidden_state[:, 0, :]
        feats.append(cls.cpu().float().numpy())
        labels.extend(lbls); paths.extend(pth)
    return np.concatenate(feats, axis=0), labels, paths


print('\nFrozen extract TRAIN...')
t0 = time.time()
X_train_deep, y_train, paths_train = extract_deep_features(train_samples, 'Train')
print(f'Done {time.time()-t0:.1f}s | shape={X_train_deep.shape}')

print('Frozen extract VAL...')
t0 = time.time()
X_val_deep, y_val, paths_val = extract_deep_features(val_samples, 'Val  ')
print(f'Done {time.time()-t0:.1f}s | shape={X_val_deep.shape}')

print('Frozen extract TEST...')
t0 = time.time()
X_test_deep, y_test, paths_test = extract_deep_features(test_samples, 'Test ')
print(f'Done {time.time()-t0:.1f}s | shape={X_test_deep.shape}')


## 4. Handcrafted features (only 136-dim)

| Block | Size | Reason to keep |
|---|---:|---|
| HSV histogram | 96 | Color distribution — an absolute representation different from the CNN |
| Gabor filter bank | 40 | Orientation-specific texture — complements DINOv2 |
| ~~LBP~~ | ~~10~~ | DROP — nearly redundant with CNN texture |
| ~~GLCM Haralick~~ | ~~16~~ | DROP — DINOv2 already encodes statistical texture |
| ~~Color Moments~~ | ~~9~~ | DROP — duplicates the HSV histogram |
| **Total** | **136** | |


In [ ]:
def hsv_histogram(img_rgb, bins=32):
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    h = cv2.calcHist([hsv], [0], None, [bins], [0, 180]).flatten()
    s = cv2.calcHist([hsv], [1], None, [bins], [0, 256]).flatten()
    v = cv2.calcHist([hsv], [2], None, [bins], [0, 256]).flatten()
    feat = np.concatenate([h, s, v])
    return feat / (feat.sum() + 1e-8)


# Precompute Gabor kernels  ───────────────────────────────
GABOR_FREQS  = (0.1, 0.2, 0.3, 0.4, 0.5)
GABOR_ANGLES = np.linspace(0, np.pi, 8, endpoint=False)
_GABOR_KERNELS = [
    cv2.getGaborKernel(ksize=(21, 21), sigma=4.0, theta=angle,
                       lambd=1.0 / freq, gamma=0.5, psi=0, ktype=cv2.CV_32F)
    for freq in GABOR_FREQS for angle in GABOR_ANGLES
]
print(f'Precomputed {len(_GABOR_KERNELS)} Gabor kernels')


def gabor_features(gray):
    g = gray.astype(np.float32) / 255.0
    return np.array([
        np.mean(np.abs(cv2.filter2D(g, cv2.CV_32F, k))) for k in _GABOR_KERNELS
    ])


def extract_handcrafted(img_path):
    img = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    img_rgb = np.array(img)
    img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    return np.concatenate([
        hsv_histogram(img_rgb),    # 96
        gabor_features(img_gray),  # 40
    ])


def extract_handcrafted_all(paths, desc='HC features', n_jobs=-1):
    return np.array(Parallel(n_jobs=n_jobs)(
        delayed(extract_handcrafted)(p) for p in tqdm(paths, desc=desc)
    ))


print('Handcrafted TRAIN...')
t0 = time.time()
X_train_hc = extract_handcrafted_all(paths_train, 'Train HC')
print(f'Done {time.time()-t0:.1f}s | shape={X_train_hc.shape}')

print('Handcrafted VAL...')
X_val_hc  = extract_handcrafted_all(paths_val,  'Val   HC')
print('Handcrafted TEST...')
X_test_hc = extract_handcrafted_all(paths_test, 'Test  HC')
print(f'Shape val={X_val_hc.shape}  test={X_test_hc.shape}')


## 5. ELM, Fusion (α=0.95), PCA adaptive
Frozen-path baseline — to have a baseline to compare against K-fold FT.

In [ ]:
class ELM:
    """Extreme Learning Machine with L2-norm H + ridge."""
    def __init__(self, n_hidden=2000, activation='relu',
                 C=1.0, random_state=42, normalize_H=True):
        self.n_hidden     = n_hidden
        self.activation   = activation
        self.C            = C
        self.random_state = random_state
        self.normalize_H  = normalize_H
        self.W = self.b = self.beta = self.classes_ = None

    def _act(self, X):
        if self.activation == 'relu':   return np.maximum(0, X)
        if self.activation == 'sigmoid': return 1.0 / (1.0 + np.exp(-np.clip(X, -500, 500)))
        if self.activation == 'tanh':   return np.tanh(X)
        return X

    def _H(self, X):
        H = self._act(X @ self.W.T + self.b)
        if self.normalize_H:
            n = np.linalg.norm(H, axis=1, keepdims=True) + 1e-8
            H = H / n
        return H

    def fit(self, X, y):
        rng = np.random.RandomState(self.random_state)
        self.classes_ = np.unique(y)
        n_cls, n_in = len(self.classes_), X.shape[1]
        limit = np.sqrt(6.0 / (n_in + self.n_hidden))
        self.W = rng.uniform(-limit, limit, (self.n_hidden, n_in))
        self.b = rng.uniform(-limit, limit, (1, self.n_hidden))
        T = np.zeros((len(y), n_cls))
        for i, c in enumerate(self.classes_):
            T[y == c, i] = 1
        H = self._H(X)
        I = np.eye(self.n_hidden)
        self.beta = np.linalg.solve(H.T @ H + I / self.C, H.T @ T)
        return self

    def predict_proba(self, X):
        S = self._H(X) @ self.beta
        e = np.exp(S - S.max(axis=1, keepdims=True))
        return e / e.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


print('ELM ready.')


In [ ]:
# Encode + L2-norm + fusion + PCA  ────────────────────────
le = LabelEncoder().fit(TARGET_CLASSES)
y_train_enc = le.transform(y_train)
y_val_enc   = le.transform(y_val)
y_test_enc  = le.transform(y_test)
print(f'Classes: {list(le.classes_)}')

X_train_deep_n = normalize(X_train_deep, norm='l2')
X_val_deep_n   = normalize(X_val_deep,   norm='l2')
X_test_deep_n  = normalize(X_test_deep,  norm='l2')
X_train_hc_n   = normalize(X_train_hc,   norm='l2')
X_val_hc_n     = normalize(X_val_hc,     norm='l2')
X_test_hc_n    = normalize(X_test_hc,    norm='l2')

# Fusion alpha = 0.95 (deep heavily favored - HC is only an auxiliary signal)
ALPHA = 0.95
X_train_raw = np.concatenate([ALPHA * X_train_deep_n, (1 - ALPHA) * X_train_hc_n], axis=1)
X_val_raw   = np.concatenate([ALPHA * X_val_deep_n,   (1 - ALPHA) * X_val_hc_n],   axis=1)
X_test_raw  = np.concatenate([ALPHA * X_test_deep_n,  (1 - ALPHA) * X_test_hc_n],  axis=1)
print(f'Fused: {X_train_raw.shape}  (alpha={ALPHA})')

# PCA adaptive  ───────────────────────────────────────────
VAR_THRESH = 0.95
MAX_PCA_DIM = 512
pca_probe = PCA(random_state=SEED, svd_solver='full').fit(X_train_raw)
cum = np.cumsum(pca_probe.explained_variance_ratio_)
PCA_DIM = int(np.searchsorted(cum, VAR_THRESH)) + 1
PCA_DIM = min(PCA_DIM, MAX_PCA_DIM, X_train_raw.shape[1])
print(f'PCA adaptive: keep {VAR_THRESH*100:.0f}% var -> {PCA_DIM} dims')

pca = PCA(n_components=PCA_DIM, random_state=SEED, svd_solver='full')
X_train = pca.fit_transform(X_train_raw)
X_val   = pca.transform(X_val_raw)
X_test  = pca.transform(X_test_raw)
print(f'X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}')


## 6. Grid Search ELM (5-fold CV F1-macro)

In [ ]:
C_grid      = [0.01, 0.1, 1, 10, 100]
hidden_grid = [1000, 2000, 3000, 4000, 5000]

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
grid_results = []
best_f1, best_C, best_H = 0.0, 1, 1000

print('Grid Search...')
total = len(C_grid) * len(hidden_grid)
pbar = tqdm(total=total, desc='Grid')
for n_h in hidden_grid:
    for C in C_grid:
        f1s = []
        for tr_idx, vl_idx in kf.split(X_train, y_train_enc):
            elm = ELM(n_hidden=n_h, C=C, random_state=SEED)
            elm.fit(X_train[tr_idx], y_train_enc[tr_idx])
            f1s.append(f1_score(y_train_enc[vl_idx],
                                elm.predict(X_train[vl_idx]),
                                average='macro'))
        m = float(np.mean(f1s))
        grid_results.append({'n_hidden': n_h, 'C': C, 'cv_f1': m})
        if m > best_f1:
            best_f1, best_C, best_H = m, C, n_h
        pbar.set_postfix(n_h=n_h, C=C, f1=f'{m:.4f}')
        pbar.update(1)
pbar.close()
print(f'\nBest: n_hidden={best_H}  C={best_C}  CV F1={best_f1:.4f}')


In [ ]:
# Ensemble ELM (10x soft voting)
N_ENSEMBLES = 10
print(f'Train Ensemble ELM ({N_ENSEMBLES}x, n_hidden={best_H}, C={best_C})...')
t0 = time.time()
ensemble_elms = []
for i in tqdm(range(N_ENSEMBLES), desc='Ensemble ELM'):
    e = ELM(n_hidden=best_H, C=best_C, random_state=SEED + i)
    e.fit(X_train, y_train_enc)
    ensemble_elms.append(e)
t_train = time.time() - t0
print(f'Done {t_train*1000:.1f} ms')


def ensemble_predict_proba(X, models):
    return np.mean([m.predict_proba(X) for m in models], axis=0)


def ensemble_predict(X, models, label_encoder):
    return label_encoder.classes_[ensemble_predict_proba(X, models).argmax(axis=1)]


# Evaluate baseline frozen path
y_pred_frozen = ensemble_predict(X_test, ensemble_elms, le)
y_true = np.array(y_test)
acc_frozen = accuracy_score(y_true, y_pred_frozen)
f1_frozen  = f1_score(y_true, y_pred_frozen, average='macro')
pre_frozen = precision_score(y_true, y_pred_frozen, average='macro')
rec_frozen = recall_score(y_true, y_pred_frozen, average='macro')

print('\n' + '='*60)
print('  BASELINE: ViT-L Frozen + HC + ELM Ensemble')
print('='*60)
print(f'  Accuracy : {acc_frozen:.4f}  ({acc_frozen*100:.2f}%)')
print(f'  Precision: {pre_frozen:.4f}')
print(f'  Recall   : {rec_frozen:.4f}')
print(f'  F1 Macro : {f1_frozen:.4f}')


## 7. Fine-tune K-fold Ensemble (5 folds × 18 epochs)

| Configuration | |
|---|---|
| Backbone | DINOv2 ViT-L/14, **last 3 blocks + LayerNorm** unfrozen |
| Head | LN → Dropout(0.3) → Linear(1024→256) → GELU → Dropout(0.225) → Linear(256→4) |
| Augment | RandomResizedCrop, RandomFlip, ColorJitter, RandAugment(2,10), Mixup(α=0.2) **or** CutMix(α=1.0) random switch, RandomErasing |
| Optimizer | AdamW backbone lr=1e-5, head lr=1e-4, WD per layer group |
| Scheduler | CosineAnnealingLR, eta_min=1e-6 |
| Regularization | LabelSmooth=0.1, GradClip=1.0, EMA decay=0.999 |
| K-fold | 5-fold StratifiedKFold on (train ∪ val), test unchanged |
| Aggregation | `mean(softmax_k for k in 1..5).argmax()` |


In [ ]:
# FT model definition  ─────────────────────────────────
class DINOv2FineTuner(nn.Module):
    def __init__(self, backbone, n_classes=4, n_unfreeze=3, p_drop=0.3):
        super().__init__()
        self.backbone = backbone
        for p in self.backbone.parameters():
            p.requires_grad = False
        total_blocks = len(self.backbone.encoder.layer)
        for i, block in enumerate(self.backbone.encoder.layer):
            if i >= total_blocks - n_unfreeze:
                for p in block.parameters():
                    p.requires_grad = True
        for p in self.backbone.layernorm.parameters():
            p.requires_grad = True

        self.head = nn.Sequential(
            nn.LayerNorm(DEEP_DIM),
            nn.Dropout(p_drop),
            nn.Linear(DEEP_DIM, 256),
            nn.GELU(),
            nn.Dropout(p_drop * 0.75),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        out = self.backbone(pixel_values=x)
        cls = out.last_hidden_state[:, 0, :]
        return self.head(cls), cls

    def get_features(self, x):
        out = self.backbone(pixel_values=x)
        return out.last_hidden_state[:, 0, :]


# Augment + dataset  ────────────────────────────────────
train_transform = T.Compose([
    T.Resize((256, 256)),
    T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0), ratio=(0.85, 1.18)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(p=0.3),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.03),
    T.RandAugment(num_ops=2, magnitude=10),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.30, scale=(0.02, 0.18), value='random'),
])
val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class CoffeeDatasetFT(Dataset):
    def __init__(self, samples, transform, label_encoder):
        self.samples, self.transform, self.le = samples, transform, label_encoder
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        return self.transform(img), int(self.le.transform([label])[0])


print('FT model class & transforms ready.')


In [ ]:
# Mixup + CutMix + Loss + EMA  ───────────────────────────
LABEL_SMOOTH = 0.1
MIXUP_ALPHA  = 0.2
CUTMIX_ALPHA = 1.0
EMA_DECAY    = 0.999


def label_smooth_loss(logits, targets, eps=LABEL_SMOOTH, n_cls=4):
    targets = targets.long()
    log_probs = F.log_softmax(logits, dim=-1)
    smooth = torch.full_like(log_probs, eps / n_cls)
    smooth.scatter_(1, targets.unsqueeze(1), 1 - eps + eps / n_cls)
    return -(smooth * log_probs).sum(dim=-1).mean()


def mixup_batch(x, y, alpha=MIXUP_ALPHA):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


def cutmix_batch(x, y, alpha=CUTMIX_ALPHA):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    B, C, H, W = x.shape
    cut_rat = np.sqrt(1.0 - lam)
    cut_h = int(H * cut_rat); cut_w = int(W * cut_rat)
    cy = np.random.randint(H); cx = np.random.randint(W)
    y1 = max(cy - cut_h // 2, 0); y2 = min(cy + cut_h // 2, H)
    x1 = max(cx - cut_w // 2, 0); x2 = min(cx + cut_w // 2, W)
    x_new = x.clone()
    x_new[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    # Adjust lam to actual ratio
    lam = 1.0 - ((y2 - y1) * (x2 - x1) / (H * W))
    return x_new, y, y[idx], lam


def mix_loss(logits, y_a, y_b, lam):
    return lam * label_smooth_loss(logits, y_a) + (1 - lam) * label_smooth_loss(logits, y_b)


class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(self.decay).add_(p.detach(), alpha=1 - self.decay)
    def apply_to(self, model):
        self._backup = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
        for n, p in model.named_parameters():
            if p.requires_grad:
                p.data.copy_(self.shadow[n])
    def restore(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad:
                p.data.copy_(self._backup[n])
        del self._backup


print('Mixup + CutMix + EMA ready.')


In [ ]:
# K-FOLD FINE-TUNE TRAINING (refactored: uses the train_one_model() helper)
trainval_samples = train_samples + val_samples
tv_paths   = [s[0] for s in trainval_samples]
tv_labels  = [s[1] for s in trainval_samples]
tv_y_enc   = le.transform(tv_labels)
tv_idx_arr = np.arange(len(trainval_samples))

# Fixed test loader (same for all folds)
ft_test_ds = CoffeeDatasetFT(test_samples, val_transform, le)
ft_test_loader = DataLoader(
    ft_test_ds, batch_size=FT_BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'),
    worker_init_fn=seed_worker, generator=GENERATOR,
)


@torch.no_grad()
def softmax_predict(model, loader):
    model.eval()
    probs, lbls = [], []
    use_amp = (DEVICE.type == 'cuda')
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=use_amp):
            logits, _ = model(imgs)
            p = F.softmax(logits, dim=-1)
        probs.append(p.cpu().float().numpy())
        lbls.extend(labels.numpy())
    return np.concatenate(probs, axis=0), np.array(lbls)


@torch.no_grad()
def evaluate_loss_acc(model, loader):
    model.eval()
    total_loss, preds, lbls = 0.0, [], []
    use_amp = (DEVICE.type == 'cuda')
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        labs = labels.to(DEVICE).long()
        with torch.amp.autocast('cuda', enabled=use_amp):
            logits, _ = model(imgs)
            loss = label_smooth_loss(logits, labs)
        total_loss += loss.item() * len(labels)
        preds.extend(logits.argmax(dim=-1).cpu().numpy())
        lbls.extend(labels.numpy())
    n = len(lbls)
    return (total_loss / n,
            accuracy_score(lbls, preds),
            f1_score(lbls, preds, average='macro'))


def train_one_model(fold_train, fold_val, *,
                    backbone_name=DINOV2_MODEL, deep_dim=DEEP_DIM,
                    n_unfreeze=3, p_drop=0.3,
                    epochs=FT_EPOCHS, patience=PATIENCE,
                    tag=''):
    """Train 1 fine-tuned DINOv2 head + last-N blocks on (fold_train, fold_val).
    Re-used by K-fold loop, single-FT ablation, and ViT-B ablation.
    Returns: best_state_dict, best_val_f1
    """
    tr_ds = CoffeeDatasetFT(fold_train, train_transform, le)
    va_ds = CoffeeDatasetFT(fold_val,   val_transform,   le)

    lbl_ids = np.array([int(le.transform([l])[0]) for _, l in fold_train])
    cls_cnt = np.bincount(lbl_ids, minlength=len(TARGET_CLASSES)).astype(np.float64)
    samp_w  = 1.0 / cls_cnt[lbl_ids]
    sampler = WeightedRandomSampler(samp_w, num_samples=len(samp_w), replacement=True)

    pin = (DEVICE.type == 'cuda')
    tr_loader = DataLoader(tr_ds, batch_size=FT_BATCH_SIZE, sampler=sampler,
                           num_workers=NUM_WORKERS, pin_memory=pin,
                           worker_init_fn=seed_worker, generator=GENERATOR)
    va_loader = DataLoader(va_ds, batch_size=FT_BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=pin,
                           worker_init_fn=seed_worker, generator=GENERATOR)

    backbone_local = AutoModel.from_pretrained(backbone_name)
    model = DINOv2FineTuner(backbone_local, n_classes=4,
                            n_unfreeze=n_unfreeze, p_drop=p_drop).to(DEVICE)
    # Override DEEP_DIM for ViT-B head if needed
    if deep_dim != DEEP_DIM:
        model.head = nn.Sequential(
            nn.LayerNorm(deep_dim), nn.Dropout(p_drop),
            nn.Linear(deep_dim, 256), nn.GELU(),
            nn.Dropout(p_drop * 0.75), nn.Linear(256, 4),
        ).to(DEVICE)

    bb_params = [p for p in model.backbone.parameters() if p.requires_grad]
    hd_params = list(model.head.parameters())
    opt = torch.optim.AdamW([
        {'params': bb_params, 'lr': 1e-5, 'weight_decay': 5e-4},
        {'params': hd_params, 'lr': 1e-4, 'weight_decay': 5e-2},
    ])
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))
    ema    = EMA(model, decay=EMA_DECAY)

    best_f1, best_state, no_imp = 0.0, None, 0
    use_amp = (DEVICE.type == 'cuda')

    print(f'{"Ep":>3} | {"TrLoss":>7} {"VaLoss":>7} | {"VaAcc":>7} {"VaF1":>7} | {"LR":>9}  [{tag}]')
    print('-' * 66)
    for epoch in range(epochs):
        model.train()
        tr_loss, total = 0.0, 0
        for imgs, labels in tr_loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            labs = labels.to(DEVICE).long()
            if np.random.rand() < 0.5:
                imgs_m, ya, yb, lam = mixup_batch(imgs, labs)
            else:
                imgs_m, ya, yb, lam = cutmix_batch(imgs, labs)
            opt.zero_grad()
            with torch.amp.autocast('cuda', enabled=use_amp):
                logits, _ = model(imgs_m)
                loss = mix_loss(logits, ya, yb, lam)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(opt); scaler.update()
            ema.update(model)
            tr_loss += loss.item() * len(labels)
            total   += len(labels)
        train_loss = tr_loss / total
        sched.step()

        ema.apply_to(model)
        va_loss, va_acc, va_f1 = evaluate_loss_acc(model, va_loader)
        ema.restore(model)

        cur_lr = opt.param_groups[1]['lr']
        flag = ''
        if va_f1 > best_f1:
            best_f1 = va_f1
            ema.apply_to(model)
            best_state = copy.deepcopy(model.state_dict())
            ema.restore(model)
            no_imp = 0; flag = '  *'
        else:
            no_imp += 1
        print(f'{epoch+1:>3} | {train_loss:>7.4f} {va_loss:>7.4f} | '
              f'{va_acc:>7.4f} {va_f1:>7.4f} | {cur_lr:>9.2e}{flag}')
        if no_imp >= patience:
            print(f'[EarlyStop {tag}] no improvement for {patience} epochs.')
            break

    del model, backbone_local, opt, sched, scaler, ema, tr_loader, va_loader
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return best_state, best_f1


# ---- Run K-fold using the helper ----
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_test_probs = []      # 5 arrays of shape (N_test, 4)
fold_dev_f1   = []
fold_states   = []
fold_test_lbls = None

for fold_i, (tr_idx, va_idx) in enumerate(skf.split(tv_idx_arr, tv_y_enc)):
    print('\n' + '#' * 70)
    print(f'#  FOLD {fold_i+1} / {N_FOLDS}')
    print('#' * 70)
    fold_train = [trainval_samples[i] for i in tr_idx]
    fold_val   = [trainval_samples[i] for i in va_idx]
    state, val_f1 = train_one_model(fold_train, fold_val, tag=f'fold-{fold_i+1}')

    # Predict on TEST with best (EMA) state
    backbone_eval = AutoModel.from_pretrained(DINOV2_MODEL)
    model_eval = DINOv2FineTuner(backbone_eval, n_classes=4, n_unfreeze=3, p_drop=0.3).to(DEVICE)
    model_eval.load_state_dict(state)
    test_probs_k, test_lbls_k = softmax_predict(model_eval, ft_test_loader)
    fold_test_probs.append(test_probs_k)
    fold_test_lbls = test_lbls_k
    fold_dev_f1.append(val_f1)
    fold_states.append(state)
    print(f'[FOLD {fold_i+1}] best Val F1 = {val_f1:.4f}')

    del model_eval, backbone_eval
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

print('\n' + '=' * 60)
print(f'K-fold training done. Per-fold dev F1: {[f"{f:.4f}" for f in fold_dev_f1]}')
print(f'Mean dev F1: {np.mean(fold_dev_f1):.4f} +/- {np.std(fold_dev_f1):.4f}')


## 8. Evaluate K-fold Ensemble — no TTA + Multi-scale TTA

In [ ]:
# K-fold ensemble (no TTA): average softmax 5 fold  ─────
avg_probs_kfold = np.mean(fold_test_probs, axis=0)
y_pred_kfold = avg_probs_kfold.argmax(axis=1)
y_true_test  = fold_test_lbls

acc_kfold = accuracy_score(y_true_test, y_pred_kfold)
f1_kfold  = f1_score(y_true_test, y_pred_kfold, average='macro')
pre_kfold = precision_score(y_true_test, y_pred_kfold, average='macro')
rec_kfold = recall_score(y_true_test, y_pred_kfold, average='macro')

print('=' * 60)
print('  K-FOLD ENSEMBLE (5 folds, no TTA)')
print('=' * 60)
print(f'  Accuracy : {acc_kfold:.4f}  ({acc_kfold*100:.2f}%)')
print(f'  Precision: {pre_kfold:.4f}')
print(f'  Recall   : {rec_kfold:.4f}')
print(f'  F1 Macro : {f1_kfold:.4f}')

# Show per-fold to see variance
print('\nPer-fold test F1 (single model on test):')
for i, p in enumerate(fold_test_probs):
    yi = p.argmax(axis=1)
    print(f'  Fold {i+1}: F1 = {f1_score(y_true_test, yi, average="macro"):.4f}')


In [ ]:
# Multi-scale TTA: 3 scales (224, 256, 288) x 4 flips = 12 passes
# Optimisation: stack 4 flips into 1 forward (4× fewer backbone passes)
TTA_SCALES = [224, 256, 288]


def make_loader_at(scale, samples):
    tf = T.Compose([
        T.Resize((scale, scale)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    ds = CoffeeDatasetFT(samples, tf, le)
    return DataLoader(ds, batch_size=FT_BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS,
                      pin_memory=(DEVICE.type == 'cuda'),
                      worker_init_fn=seed_worker, generator=GENERATOR)


@torch.no_grad()
def predict_tta_for_model(model, samples):
    """Multi-scale + 4-flip TTA. Stack 4 flips into 1 batch -> 4x faster."""
    model.eval()
    use_amp = (DEVICE.type == 'cuda')
    all_probs = None
    for scale in TTA_SCALES:
        loader = make_loader_at(scale, samples)
        scale_probs = []
        for imgs, _ in tqdm(loader, desc=f'TTA scale={scale}', leave=False):
            imgs = imgs.to(DEVICE, non_blocking=True)
            B = imgs.size(0)
            stacked = torch.cat([
                imgs,
                torch.flip(imgs, dims=[3]),
                torch.flip(imgs, dims=[2]),
                torch.flip(imgs, dims=[2, 3]),
            ], dim=0)  # (4*B, C, H, W)
            with torch.amp.autocast('cuda', enabled=use_amp):
                logits, _ = model(stacked)
                p = F.softmax(logits, dim=-1)
            p = p.view(4, B, -1).mean(dim=0)  # average over flips
            scale_probs.append(p.cpu().float().numpy())
        scale_probs = np.concatenate(scale_probs, axis=0)
        all_probs = scale_probs if all_probs is None else (all_probs + scale_probs)
    return all_probs / len(TTA_SCALES)


# Build a temporary model and run TTA for each fold
print(f'Multi-scale TTA: {TTA_SCALES} x 4 flips/fold x {N_FOLDS} folds...')
fold_test_probs_tta = []
backbone_tta = AutoModel.from_pretrained(DINOV2_MODEL)
model_tta = DINOv2FineTuner(backbone_tta, n_classes=4, n_unfreeze=3, p_drop=0.3).to(DEVICE)

for fi, state in enumerate(fold_states):
    print(f'\n[Fold {fi+1}/{N_FOLDS}] Loading & running TTA...')
    model_tta.load_state_dict(state)
    probs = predict_tta_for_model(model_tta, test_samples)
    fold_test_probs_tta.append(probs)

del model_tta, backbone_tta
gc.collect()
if DEVICE.type == 'cuda': torch.cuda.empty_cache()

# Ensemble: average across folds
avg_probs_kfold_tta = np.mean(fold_test_probs_tta, axis=0)
y_pred_kfold_tta = avg_probs_kfold_tta.argmax(axis=1)

acc_tta = accuracy_score(y_true_test, y_pred_kfold_tta)
f1_tta  = f1_score(y_true_test, y_pred_kfold_tta, average='macro')
pre_tta = precision_score(y_true_test, y_pred_kfold_tta, average='macro')
rec_tta = recall_score(y_true_test, y_pred_kfold_tta, average='macro')

print('\n' + '=' * 60)
print('  K-FOLD ENSEMBLE + MULTI-SCALE TTA (12 views/img, stacked)')
print('=' * 60)
print(f'  Accuracy : {acc_tta:.4f}  ({acc_tta*100:.2f}%)')
print(f'  Precision: {pre_tta:.4f}')
print(f'  Recall   : {rec_tta:.4f}')
print(f'  F1 Macro : {f1_tta:.4f}')

# Set "best" path = K-fold + TTA
y_proba_test = avg_probs_kfold_tta
y_pred_test  = y_pred_kfold_tta


## 9. Confusion matrix & per-class metrics (K-fold + TTA)

In [ ]:
target_names = [CLASS_VI[c] for c in le.classes_]
y_true_str = le.classes_[y_true_test]
y_pred_str = le.classes_[y_pred_test]

print(classification_report(y_true_str, y_pred_str, target_names=target_names))

cm = confusion_matrix(y_true_str, y_pred_str, labels=le.classes_)
cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, fmt, title in zip(axes, [cm, cm_n], ['d', '.2f'],
                                ['Counts', 'Normalized']):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues', ax=ax,
                xticklabels=target_names, yticklabels=target_names, linewidths=0.5)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.suptitle(f'K-fold Ensemble + TTA  |  Acc={acc_tta:.4f}  F1={f1_tta:.4f}',
             fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/coffee_outputs/fig_confusion_kfold_tta.png', dpi=160, bbox_inches='tight')
plt.show()


## 9.1 Bootstrap CI (95%) + McNemar exact test

The test set has only ~945 images, so top-tier numbers differing by ~2 images may be within noise. The two statistical tools below address this:

* **Bootstrap 95% CI** (1000 resamples) for every metric — report a CI instead of a single point.
* **McNemar exact test** between K-fold (no-TTA) and K-fold+TTA — test whether the difference is truly significant.


In [ ]:
# Bootstrap CI + McNemar  ─────────────────────────────
def bootstrap_metric(y_true, y_pred, fn, n=N_BOOTSTRAP, seed=SEED):
    """Bootstrap 95% CI for any metric fn(y_true_b, y_pred_b)."""
    rng = np.random.RandomState(seed)
    N = len(y_true)
    vals = np.empty(n, dtype=np.float64)
    for i in range(n):
        idx = rng.randint(0, N, size=N)
        vals[i] = fn(y_true[idx], y_pred[idx])
    point = fn(y_true, y_pred)
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return point, lo, hi


def acc_fn(yt, yp): return accuracy_score(yt, yp)
def f1_fn (yt, yp): return f1_score(yt, yp, average='macro', zero_division=0)


def mcnemar_exact(y_true, y_pred_a, y_pred_b):
    """Exact McNemar (binomial) — significance between 2 classifiers on the same test set."""
    a_ok = (y_pred_a == y_true)
    b_ok = (y_pred_b == y_true)
    n01 = int(((~a_ok) & (b_ok)).sum())   # A wrong, B correct
    n10 = int((( a_ok) & (~b_ok)).sum())  # A correct, B wrong
    n_disc = n01 + n10
    if n_disc == 0:
        return n01, n10, 1.0
    # Two-sided binomial p-value with p=0.5
    k = min(n01, n10)
    p = 2 * sps.binom.cdf(k, n_disc, 0.5)
    p = min(p, 1.0)
    return n01, n10, float(p)


# Wrap K-fold and K-fold+TTA preds in encoded ints (already in y_pred_kfold / y_pred_kfold_tta)
yt = np.asarray(y_true_test)

print('=' * 78)
print(f'  BOOTSTRAP 95% CI ({N_BOOTSTRAP} resamples) — top methods')
print('=' * 78)
print(f'  {"Method":<46} {"Acc [95% CI]":<22} {"F1  [95% CI]":<22}')
print('-' * 78)

ci_table = {}
top_preds = {
    'Frozen + Ens ELM' : le.transform(y_pred_frozen),
    'K-fold (no TTA)'  : y_pred_kfold,
    'K-fold + TTA *'   : y_pred_kfold_tta,
}
for name, yp in top_preds.items():
    a, alo, ahi = bootstrap_metric(yt, yp, acc_fn)
    f, flo, fhi = bootstrap_metric(yt, yp, f1_fn)
    ci_table[name] = dict(acc=a, acc_lo=alo, acc_hi=ahi, f1=f, f1_lo=flo, f1_hi=fhi)
    print(f'  {name:<46} {a:.4f}[{alo:.4f},{ahi:.4f}]  '
          f'{f:.4f}[{flo:.4f},{fhi:.4f}]')

print('\nMcNemar exact test (2-tailed):')
for (a, b) in [('Frozen + Ens ELM', 'K-fold (no TTA)'),
               ('K-fold (no TTA)', 'K-fold + TTA *')]:
    pa = top_preds[a]; pb = top_preds[b]
    n01, n10, pv = mcnemar_exact(yt, pa, pb)
    sig = 'SIG  (p<0.05)' if pv < 0.05 else 'n.s.'
    print(f'  {a:<22} vs  {b:<22} : '
          f'A->B gain={n01}, A->B loss={n10}, p={pv:.4f}  [{sig}]')


## 9.2 Phoma deep-dive — the weakest class

Phoma has the lowest Precision and F1. Detailed analysis:
* FN/FP breakdown (which classes is Phoma confused with? which classes get confused as Phoma?)
* Confusion sub-matrix Phoma↔other classes
* Top-8 hardest Phoma errors with confidence
* Confidence histogram for correct vs wrong Phoma


In [ ]:
# Phoma deep-dive  ─────────────────────────────────────
PHOMA_IDX = list(le.classes_).index('Phoma')
yt_arr = np.asarray(y_true_test)
yp_arr = np.asarray(y_pred_test)
proba  = y_proba_test

# Confusion sub-matrix (rows=true, cols=pred)
mask_phoma_true = (yt_arr == PHOMA_IDX)
mask_phoma_pred = (yp_arr == PHOMA_IDX)

# FN: true=Phoma, pred=other
fn_idx = np.where(mask_phoma_true & (~mask_phoma_pred))[0]
# FP: pred=Phoma, true=other
fp_idx = np.where((~mask_phoma_true) & mask_phoma_pred)[0]
tp_idx = np.where(mask_phoma_true & mask_phoma_pred)[0]

print(f'Phoma TP={len(tp_idx)}  FN={len(fn_idx)}  FP={len(fp_idx)}')

# FN -> what classes did the model predict?
print('\nPhoma FN: true=Phoma, predicted as ->')
fn_pred_dist = Counter(yp_arr[fn_idx].tolist())
for c, n in fn_pred_dist.most_common():
    print(f'  {TARGET_CLASSES[c]:<12}: {n:>4}')

# FP -> which true classes were misclassified as Phoma?
print('\nPhoma FP: predicted=Phoma, true ->')
fp_true_dist = Counter(yt_arr[fp_idx].tolist())
for c, n in fp_true_dist.most_common():
    print(f'  {TARGET_CLASSES[c]:<12}: {n:>4}')

# Confidence histogram: Phoma correct vs wrong
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
if len(tp_idx):
    ax.hist(proba[tp_idx, PHOMA_IDX], bins=20, range=(0, 1),
            color='#4CAF50', alpha=0.85, label=f'Phoma OK (n={len(tp_idx)})',
            edgecolor='black', linewidth=0.3)
if len(fn_idx):
    # confidence of WRONG prediction (the predicted class softmax)
    wrong_conf = proba[fn_idx, yp_arr[fn_idx]]
    ax.hist(wrong_conf, bins=20, range=(0, 1),
            color='#d62728', alpha=0.85, label=f'Phoma -> other (FN, n={len(fn_idx)})',
            edgecolor='black', linewidth=0.3)
ax.set_xlabel('Confidence (max softmax)')
ax.set_ylabel('Count')
ax.set_title('Phoma — confidence: correct vs misclassified', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)

# Per-class confusion bar (Phoma related only)
ax = axes[1]
labels_pl, vals = [], []
for c in range(4):
    if c == PHOMA_IDX: continue
    fn_c = int((yt_arr[fn_idx] == PHOMA_IDX).sum() if False else (yp_arr[fn_idx] == c).sum())
    fp_c = int((yt_arr[fp_idx] == c).sum())
    labels_pl.append(TARGET_CLASSES[c])
    vals.append([fn_c, fp_c])
vals = np.array(vals)
x = np.arange(len(labels_pl)); w = 0.4
ax.bar(x - w/2, vals[:, 0], w, label='Phoma -> X (FN)', color='#d62728', alpha=0.85)
ax.bar(x + w/2, vals[:, 1], w, label='X -> Phoma (FP)', color='#FF9800', alpha=0.85)
for i, (a, b) in enumerate(vals):
    ax.text(i - w/2, a + 0.2, str(a), ha='center', fontsize=9)
    ax.text(i + w/2, b + 0.2, str(b), ha='center', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(labels_pl)
ax.set_ylabel('# images')
ax.set_title('Phoma confusion: where errors go', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/coffee_outputs/fig_phoma_conf.png', dpi=160, bbox_inches='tight')
plt.show()


# Top-8 hardest Phoma errors (highest confidence in WRONG class)
hard_idx = np.concatenate([fn_idx, fp_idx]) if len(fn_idx) + len(fp_idx) > 0 else np.array([], dtype=int)
n_show = min(8, len(hard_idx))
if n_show > 0:
    confs = proba[hard_idx].max(axis=1)
    order = np.argsort(-confs)[:n_show]
    show_idx = hard_idx[order]

    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for ax, i in zip(axes.flat, show_idx):
        try:
            img = Image.open(test_samples[i][0]).convert('RGB').resize((224, 224))
            ax.imshow(img)
        except Exception: pass
        ax.axis('off')
        t = TARGET_CLASSES[yt_arr[i]]
        p = TARGET_CLASSES[yp_arr[i]]
        ax.set_title(f'T:{t} -> P:{p}\nconf={proba[i].max()*100:.1f}%',
                     color='#d62728', fontsize=10)
    for ax in axes.flat[n_show:]:
        ax.axis('off')
    fig.suptitle('Phoma — top hardest errors (FN + FP, highest wrong confidence)',
                 fontweight='bold')
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/coffee_outputs/fig_phoma_errors.png', dpi=160, bbox_inches='tight')
    plt.show()

# Numeric summary
phoma_p = precision_score(yt_arr, yp_arr, labels=[PHOMA_IDX], average='macro', zero_division=0)
phoma_r = recall_score   (yt_arr, yp_arr, labels=[PHOMA_IDX], average='macro', zero_division=0)
phoma_f = f1_score       (yt_arr, yp_arr, labels=[PHOMA_IDX], average='macro', zero_division=0)
print(f'\nPhoma — P={phoma_p:.4f}  R={phoma_r:.4f}  F1={phoma_f:.4f}  '
      f'(support={int(mask_phoma_true.sum())})')


## 10. Comparison table of all methods (v2)

In [ ]:
compare = {}

def _add(name, preds, t_ms, features):
    compare[name] = {
        'acc': accuracy_score(y_true, preds),
        'f1' : f1_score(y_true, preds, average='macro'),
        'pre': precision_score(y_true, preds, average='macro', zero_division=0),
        'rec': recall_score(y_true, preds, average='macro', zero_division=0),
        'ms' : t_ms,
        'features': features,
    }


# 1. ViT-L Frozen + HC + SVM
print('[1] ViT-L Frozen + HC + SVM...')
svm1 = SVC(kernel='rbf', C=10, probability=True, random_state=SEED)
t0 = time.time(); svm1.fit(X_train, y_train_enc); t1 = time.time() - t0
_add('ViT-L Frozen+HC+PCA + SVM',
     le.inverse_transform(svm1.predict(X_test)), t1*1000, 'Frozen+HC+PCA')

# 2. ViT-L Frozen + HC + ELM single
print('[2] ViT-L Frozen + HC + ELM (single)...')
elm_single = ELM(n_hidden=best_H, C=best_C, random_state=SEED)
t0 = time.time(); elm_single.fit(X_train, y_train_enc); t2 = time.time() - t0
_add('ViT-L Frozen+HC+PCA + ELM',
     le.inverse_transform(elm_single.predict(X_test)), t2*1000, 'Frozen+HC+PCA')

# 3. ViT-L Frozen + HC + Ensemble ELM
_add('ViT-L Frozen+HC+PCA + Ensemble ELM (10x)',
     y_pred_frozen, t_train*1000, 'Frozen+HC+PCA')

# 4. K-fold (no TTA)
_add('ViT-L K-fold Ensemble (5 folds, no TTA)',
     le.inverse_transform(y_pred_kfold), 0, 'FT 5-fold')

# 5. K-fold + multi-scale TTA  ★ PROPOSED
_add('ViT-L K-fold + Multi-scale TTA *',
     le.inverse_transform(y_pred_kfold_tta), 0, 'FT 5-fold + TTA')

# 6. Random Forest baseline
from sklearn.ensemble import RandomForestClassifier
print('[6] Random Forest...')
t0 = time.time()
rf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED)
rf.fit(X_train, y_train_enc); t_rf = time.time() - t0
_add('ViT-L Frozen+HC+PCA + Random Forest',
     le.inverse_transform(rf.predict(X_test)), t_rf*1000, 'Frozen+HC+PCA')

# 7. XGBoost baseline
import xgboost as xgb
print('[7] XGBoost...')
t0 = time.time()
xgb_clf = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric='mlogloss',
    tree_method='hist', device='cuda' if DEVICE.type == 'cuda' else 'cpu',
    random_state=SEED, n_jobs=-1, verbosity=0,
)
xgb_clf.fit(X_train, y_train_enc); t_xgb = time.time() - t0
_add('ViT-L Frozen+HC+PCA + XGBoost',
     le.inverse_transform(xgb_clf.predict(X_test)), t_xgb*1000, 'Frozen+HC+PCA')

print('\nDone all methods.')


In [ ]:
# Sorted comparison table + 95% bootstrap CI  ─────────
def _bootstrap_for_method(name, preds_str_or_int):
    """Accept preds in str or int. Returns (acc, acc_ci, f1, f1_ci)."""
    yp = preds_str_or_int
    if isinstance(yp[0], (str, np.str_)):
        ytm = np.asarray(y_true)
        ypm = np.asarray(yp)
    else:
        ytm = np.asarray(y_true_test)
        ypm = np.asarray(yp)
    a, alo, ahi = bootstrap_metric(ytm, ypm, acc_fn)
    f, flo, fhi = bootstrap_metric(ytm, ypm, f1_fn)
    return a, (alo, ahi), f, (flo, fhi)


# Pre-compute CI for every method (uses preds we stashed in compare[*])
method_preds = {
    'ViT-L Frozen+HC+PCA + SVM'                 : le.inverse_transform(svm1.predict(X_test)),
    'ViT-L Frozen+HC+PCA + ELM'                 : le.inverse_transform(elm_single.predict(X_test)),
    'ViT-L Frozen+HC+PCA + Ensemble ELM (10x)'  : y_pred_frozen,
    'ViT-L K-fold Ensemble (5 folds, no TTA)'   : le.inverse_transform(y_pred_kfold),
    'ViT-L K-fold + Multi-scale TTA *'          : le.inverse_transform(y_pred_kfold_tta),
    'ViT-L Frozen+HC+PCA + Random Forest'       : le.inverse_transform(rf.predict(X_test)),
    'ViT-L Frozen+HC+PCA + XGBoost'             : le.inverse_transform(xgb_clf.predict(X_test)),
}

print('Computing bootstrap 95% CI for every method...')
for name in compare.keys():
    if name in method_preds:
        a, (alo, ahi), f, (flo, fhi) = _bootstrap_for_method(name, method_preds[name])
        compare[name].update(acc=a, acc_lo=alo, acc_hi=ahi,
                             f1=f, f1_lo=flo, f1_hi=fhi)

sorted_compare = dict(sorted(compare.items(), key=lambda x: x[1]['acc'], reverse=True))

print('\n' + '=' * 116)
print(f'  {"COMPREHENSIVE COMPARISON TABLE — v2.1  (95% bootstrap CI, " + str(N_BOOTSTRAP) + " resamples)":^112}')
print('=' * 116)
print(f'  {"#":>3}  {"Method":<48} {"Features":<18} '
      f'{"Acc [95% CI]":>22} {"F1 [95% CI]":>22}')
print('-' * 116)
for rank, (m, r) in enumerate(sorted_compare.items(), 1):
    star = ' *' if rank == 1 else '  '
    acc_s = f'{r["acc"]:.4f}[{r.get("acc_lo", r["acc"]):.4f},{r.get("acc_hi", r["acc"]):.4f}]'
    f1_s  = f'{r["f1"] :.4f}[{r.get("f1_lo",  r["f1"] ):.4f},{r.get("f1_hi",  r["f1"] ):.4f}]'
    print(f'  {rank:>3}{star}  {m:<48} {r["features"]:<18} {acc_s:>22} {f1_s:>22}')
print('=' * 116)
best_m = list(sorted_compare.keys())[0]
print(f'\n  * BEST: {best_m}')
print(f'    Accuracy : {sorted_compare[best_m]["acc"]:.4f} '
      f'({sorted_compare[best_m]["acc"]*100:.2f}%)')
print(f'    F1 Macro : {sorted_compare[best_m]["f1"]:.4f}')

# Save CSV with CI
rows = []
for name, r in sorted_compare.items():
    rows.append({
        'method': name,
        'features': r['features'],
        'acc': r['acc'], 'acc_lo': r.get('acc_lo'), 'acc_hi': r.get('acc_hi'),
        'f1':  r['f1'],  'f1_lo' : r.get('f1_lo'),  'f1_hi':  r.get('f1_hi'),
        'precision': r['pre'], 'recall': r['rec'],
    })
pd.DataFrame(rows).to_csv('/content/drive/MyDrive/coffee_outputs/methods_ci_v2_1.csv', index=False)
print('\n[OK] methods_ci_v2_1.csv saved')


In [ ]:
# Bar chart comparison  ────────────────────────────────
def get_color(name):
    if 'TTA' in name:               return '#E53935'
    if 'K-fold' in name:            return '#FF9800'
    if 'Ensemble ELM' in name:      return '#FFB74D'
    if 'ELM' in name:               return '#FFCA28'
    if 'SVM' in name:               return '#2196F3'
    if 'XGBoost' in name:           return '#9C27B0'
    if 'Random Forest' in name:     return '#4CAF50'
    return '#90A4AE'

methods_s = list(sorted_compare.keys())
accs_s = [sorted_compare[m]['acc'] for m in methods_s]
f1s_s  = [sorted_compare[m]['f1']  for m in methods_s]
colors_s = [get_color(m) for m in methods_s]
n = len(methods_s)

fig, ax = plt.subplots(figsize=(16, 6.5))
x, w = np.arange(n), 0.38
b1 = ax.bar(x - w/2, accs_s, w, label='Accuracy', color=colors_s, alpha=0.95)
b2 = ax.bar(x + w/2, f1s_s,  w, label='F1 Macro', color=colors_s, alpha=0.5,
            edgecolor='grey', linewidth=0.8)
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.3f}', ha='center', fontsize=8, rotation=80)
ax.set_xticks(x)
ax.set_xticklabels(methods_s, rotation=25, ha='right', fontsize=8.5)
ax.set_ylim(0, max(accs_s + f1s_s) + 0.06)
ax.set_ylabel('Score'); ax.legend()
ax.set_title('Method comparison — Pipeline v2 (DINOv2 ViT-L + K-fold + TTA)',
             fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/coffee_outputs/fig_comparison_v2.png', dpi=160, bbox_inches='tight')
plt.show()


## 10.1 Inference speed comparison across models

The accuracy table in cell 10 shows K-fold + TTA is the most accurate, but **speed** is an important factor when deploying. This section benchmarks the inference time of ALL models on the full test set:

* **Frozen-path classifiers** (SVM, ELM, RF, XGBoost): input is PCA features → latency of the classification layer only (assuming features are already extracted)
* **Deep models** (K-fold FT, FT+TTA): input is raw images → END-TO-END latency (backbone + head)

Reports 3 metrics: latency (ms/sample), throughput (samples/sec), and the accuracy vs speed trade-off (Pareto frontier).


In [ ]:
# INFERENCE SPEED COMPARISON  ───────────────────────────────
# Distinguish 2 groups:
#   A) Frozen-path classifiers: input is PCA features (X_test)
#      - Latency of the classification layer only (post feature-extraction)
#   B) Deep models (FT): input is images -> end-to-end
#      - Full-pipeline latency (backbone + head)
#
# Reports:
#   - Total inference time on the full test set
#   - Latency = ms / sample
#   - Throughput = samples / sec
#   - Bonus: Accuracy vs Speed scatter (trade-off)

import time as _time_speed

SPEED_RUNS = 3                                  # number of runs to average
N_TEST = len(y_true_test)
speed_results = {}                              # method -> dict(total_s, ms_per_sample, samples_per_sec, acc, f1, kind)


def _bench_sklearn(name, predict_fn, acc, f1, kind='Frozen-clf'):
    # warmup
    _ = predict_fn()
    # time
    ts = []
    for _ in range(SPEED_RUNS):
        s = _time_speed.perf_counter()
        _ = predict_fn()
        ts.append(_time_speed.perf_counter() - s)
    tot = float(np.mean(ts))
    speed_results[name] = dict(
        total_s=tot,
        ms_per_sample=tot / N_TEST * 1000,
        samples_per_sec=N_TEST / tot,
        acc=acc, f1=f1, kind=kind,
    )
    print(f'  {name:<42}  tot={tot:6.3f}s  '
          f'{tot / N_TEST * 1000:7.3f} ms/img  {N_TEST / tot:8.1f} img/s')


print('=' * 80)
print('  [A] FROZEN-PATH CLASSIFIERS (input = PCA features)')
print('=' * 80)
_bench_sklearn('ViT-L Frozen+HC+PCA + SVM',
               lambda: svm1.predict(X_test),
               compare['ViT-L Frozen+HC+PCA + SVM']['acc'],
               compare['ViT-L Frozen+HC+PCA + SVM']['f1'])
_bench_sklearn('ViT-L Frozen+HC+PCA + ELM',
               lambda: elm_single.predict(X_test),
               compare['ViT-L Frozen+HC+PCA + ELM']['acc'],
               compare['ViT-L Frozen+HC+PCA + ELM']['f1'])
_bench_sklearn('ViT-L Frozen+HC+PCA + Ensemble ELM (10x)',
               lambda: ensemble_predict(X_test, ensemble_elms, le),
               compare['ViT-L Frozen+HC+PCA + Ensemble ELM (10x)']['acc'],
               compare['ViT-L Frozen+HC+PCA + Ensemble ELM (10x)']['f1'])
_bench_sklearn('ViT-L Frozen+HC+PCA + Random Forest',
               lambda: rf.predict(X_test),
               compare['ViT-L Frozen+HC+PCA + Random Forest']['acc'],
               compare['ViT-L Frozen+HC+PCA + Random Forest']['f1'])
_bench_sklearn('ViT-L Frozen+HC+PCA + XGBoost',
               lambda: xgb_clf.predict(X_test),
               compare['ViT-L Frozen+HC+PCA + XGBoost']['acc'],
               compare['ViT-L Frozen+HC+PCA + XGBoost']['f1'])


print('\n' + '=' * 80)
print('  [B] DEEP MODELS (input = raw images, end-to-end)')
print('=' * 80)

# Load 1 fold to measure end-to-end time (backbone + head)
backbone_bench = AutoModel.from_pretrained(DINOV2_MODEL)
model_bench = DINOv2FineTuner(backbone_bench, n_classes=4, n_unfreeze=3, p_drop=0.3).to(DEVICE)
model_bench.load_state_dict(fold_states[0])
model_bench.eval()


def _sync():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()


# Warmup
_ = softmax_predict(model_bench, ft_test_loader)

# (B1) Single-fold FT (no TTA)
_sync()
ts = []
for _ in range(SPEED_RUNS):
    _sync(); s = _time_speed.perf_counter()
    _ = softmax_predict(model_bench, ft_test_loader)
    _sync(); ts.append(_time_speed.perf_counter() - s)
t_single = float(np.mean(ts))
speed_results['ViT-L Single FT (no TTA, no K-fold)'] = dict(
    total_s=t_single,
    ms_per_sample=t_single / N_TEST * 1000,
    samples_per_sec=N_TEST / t_single,
    acc=None, f1=None, kind='Deep-end2end',
)
print(f'  {"Single FT (1 fold, no TTA)":<42}  tot={t_single:6.3f}s  '
      f'{t_single / N_TEST * 1000:7.3f} ms/img  {N_TEST / t_single:8.1f} img/s')

# (B2) K-fold (no TTA): 5x single-fold latency (sequential forward over 5 fold)
t_kfold_total = N_FOLDS * t_single
speed_results['ViT-L K-fold Ensemble (5 folds, no TTA)'] = dict(
    total_s=t_kfold_total,
    ms_per_sample=t_kfold_total / N_TEST * 1000,
    samples_per_sec=N_TEST / t_kfold_total,
    acc=compare['ViT-L K-fold Ensemble (5 folds, no TTA)']['acc'],
    f1=compare['ViT-L K-fold Ensemble (5 folds, no TTA)']['f1'],
    kind='Deep-end2end',
)
print(f'  {"K-fold (5 folds, no TTA)":<42}  tot={t_kfold_total:6.3f}s  '
      f'{t_kfold_total / N_TEST * 1000:7.3f} ms/img  {N_TEST / t_kfold_total:8.1f} img/s   (= 5 x single FT)')

# (B3) K-fold + Multi-scale TTA: measure 1 fold directly then multiply by 5
_sync()
ts = []
for _ in range(max(1, SPEED_RUNS - 1)):                                  # TTA is expensive - only run twice
    _sync(); s = _time_speed.perf_counter()
    _ = predict_tta_for_model(model_bench, test_samples)
    _sync(); ts.append(_time_speed.perf_counter() - s)
t_tta_fold = float(np.mean(ts))
t_kfold_tta = N_FOLDS * t_tta_fold
speed_results['ViT-L K-fold + Multi-scale TTA *'] = dict(
    total_s=t_kfold_tta,
    ms_per_sample=t_kfold_tta / N_TEST * 1000,
    samples_per_sec=N_TEST / t_kfold_tta,
    acc=compare['ViT-L K-fold + Multi-scale TTA *']['acc'],
    f1=compare['ViT-L K-fold + Multi-scale TTA *']['f1'],
    kind='Deep-end2end',
)
print(f'  {"K-fold + TTA (5 folds x 3 scales x 4 flips)":<42}  tot={t_kfold_tta:6.3f}s  '
      f'{t_kfold_tta / N_TEST * 1000:7.3f} ms/img  {N_TEST / t_kfold_tta:8.1f} img/s')

# Cleanup bench model
del model_bench, backbone_bench
gc.collect()
if DEVICE.type == 'cuda': torch.cuda.empty_cache()


# ---- (1) Bar chart: ms/sample va img/s --------------------
methods_speed = list(speed_results.keys())
lat = [speed_results[m]['ms_per_sample'] for m in methods_speed]
thr = [speed_results[m]['samples_per_sec'] for m in methods_speed]
short_lbl = [m.replace('ViT-L ', '').replace('Frozen+HC+PCA + ', '').replace(' (no K-fold)', '')
             for m in methods_speed]
colors_sp = ['#E53935' if 'TTA *' in m else
             '#9C27B0' if 'Single FT' in m else
             '#FF9800' if 'K-fold' in m else
             '#1976D2' for m in methods_speed]

fig_sp, axes_sp = plt.subplots(1, 2, figsize=(18, 6.5))

bars1 = axes_sp[0].barh(range(len(methods_speed)), lat, color=colors_sp, alpha=0.9,
                        edgecolor='black', linewidth=0.6)
axes_sp[0].set_yticks(range(len(methods_speed)))
axes_sp[0].set_yticklabels(short_lbl, fontsize=9)
axes_sp[0].set_xlabel('Latency (ms / sample)', fontweight='bold')
axes_sp[0].set_title('Inference latency (lower = faster)', fontweight='bold', fontsize=12)
axes_sp[0].set_xscale('log')
axes_sp[0].grid(axis='x', alpha=0.3, which='both')
for b, v in zip(bars1, lat):
    axes_sp[0].text(v * 1.05, b.get_y() + b.get_height() / 2, f'{v:.2f} ms',
                    va='center', fontsize=9, fontweight='bold')

bars2 = axes_sp[1].barh(range(len(methods_speed)), thr, color=colors_sp, alpha=0.9,
                        edgecolor='black', linewidth=0.6)
axes_sp[1].set_yticks(range(len(methods_speed)))
axes_sp[1].set_yticklabels(short_lbl, fontsize=9)
axes_sp[1].set_xlabel('Throughput (samples / sec)', fontweight='bold')
axes_sp[1].set_title('Throughput (higher = faster)', fontweight='bold', fontsize=12)
axes_sp[1].set_xscale('log')
axes_sp[1].grid(axis='x', alpha=0.3, which='both')
for b, v in zip(bars2, thr):
    axes_sp[1].text(v * 1.05, b.get_y() + b.get_height() / 2, f'{v:.0f}',
                    va='center', fontsize=9, fontweight='bold')

fig_sp.suptitle('Inference speed comparison across models — Pipeline v2.1',
                fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/coffee_outputs/fig_inference_speed.png', dpi=160, bbox_inches='tight')
plt.show()


# ---- (2) Accuracy vs Speed trade-off (only methods with accuracy) ----
fig_tr, ax_tr = plt.subplots(figsize=(12, 7))
plot_x, plot_y, plot_lbl, plot_col, plot_size = [], [], [], [], []
for m in methods_speed:
    r = speed_results[m]
    if r['acc'] is None:
        continue
    plot_x.append(r['samples_per_sec'])
    plot_y.append(r['acc'])
    plot_lbl.append(m)
    plot_col.append('#E53935' if 'TTA *' in m else
                    '#9C27B0' if 'Single FT' in m else
                    '#FF9800' if 'K-fold' in m else
                    '#1976D2')
    plot_size.append(280)

ax_tr.scatter(plot_x, plot_y, s=plot_size, c=plot_col, alpha=0.78,
              edgecolor='black', linewidth=1.2)
ax_tr.set_xscale('log')
for x, y, lbl in zip(plot_x, plot_y, plot_lbl):
    short = lbl.replace('ViT-L ', '').replace('Frozen+HC+PCA + ', '')
    ax_tr.annotate(short, (x, y), xytext=(7, 7), textcoords='offset points',
                   fontsize=8.5, fontweight='bold')
ax_tr.set_xlabel('Throughput (samples / sec, log scale) -->  faster', fontweight='bold')
ax_tr.set_ylabel('Accuracy -->  more accurate', fontweight='bold')
ax_tr.set_title('Trade-off: Accuracy vs Inference speed\n'
                '(top-right = ideal: accurate + fast)',
                fontweight='bold', fontsize=12)
ax_tr.grid(alpha=0.35, which='both')

# Pareto frontier highlight
sorted_by_speed = sorted(zip(plot_x, plot_y, plot_lbl), key=lambda t: t[0], reverse=True)
pareto_x, pareto_y = [], []
cur_best = -1
for x, y, l in sorted_by_speed:
    if y > cur_best:
        pareto_x.append(x); pareto_y.append(y); cur_best = y
ax_tr.plot(pareto_x, pareto_y, '--', color='gray', linewidth=1.2, alpha=0.6, label='Pareto frontier')
ax_tr.legend(loc='lower left', fontsize=10)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/coffee_outputs/fig_speed_accuracy_tradeoff.png', dpi=160, bbox_inches='tight')
plt.show()


# ---- (3) Table CSV --------------------------------------
print('\n' + '=' * 100)
print(f'  {"INFERENCE SPEED TABLE":^96}')
print('=' * 100)
print(f'  {"Method":<46} {"Kind":<14} {"ms/img":>9} {"img/s":>10} {"Acc":>8} {"F1":>8}')
print('-' * 100)
for m, r in sorted(speed_results.items(), key=lambda x: x[1]['ms_per_sample']):
    acc_s = f'{r["acc"]:.4f}' if r['acc'] is not None else '   -   '
    f1_s = f'{r["f1"]:.4f}' if r['f1'] is not None else '   -   '
    print(f'  {m:<46} {r["kind"]:<14} {r["ms_per_sample"]:>9.3f} {r["samples_per_sec"]:>10.1f} '
          f'{acc_s:>8} {f1_s:>8}')
print('=' * 100)

speed_rows = []
for m, r in speed_results.items():
    speed_rows.append(dict(
        method=m, kind=r['kind'],
        total_seconds=r['total_s'], ms_per_sample=r['ms_per_sample'],
        samples_per_sec=r['samples_per_sec'],
        accuracy=r['acc'], f1_macro=r['f1'],
    ))
pd.DataFrame(speed_rows).to_csv('/content/drive/MyDrive/coffee_outputs/inference_speed_v2_1.csv', index=False)
print('\n[OK] inference_speed_v2_1.csv saved')
print('[OK] fig_inference_speed.png saved')
print('[OK] fig_speed_accuracy_tradeoff.png saved')
print('\nNote: frozen-path classifiers measure post-feature-extraction latency.')
print('       For true end-to-end latency, add ~30-60 ms of backbone forward.')


## 11. ROC / PR / Calibration / Hardest cases (K-fold + TTA)

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

n_cls = len(TARGET_CLASSES)
y_bin = label_binarize(y_true_test, classes=list(range(n_cls)))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
for i, (cls, color) in enumerate(zip(TARGET_CLASSES, COLORS)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba_test[:, i])
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{cls}  (AUC={auc(fpr, tpr):.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC — one-vs-rest'); ax.legend(loc='lower right'); ax.grid(alpha=0.3)

ax = axes[1]
for i, (cls, color) in enumerate(zip(TARGET_CLASSES, COLORS)):
    pr, rc, _ = precision_recall_curve(y_bin[:, i], y_proba_test[:, i])
    ap = average_precision_score(y_bin[:, i], y_proba_test[:, i])
    ax.plot(rc, pr, color=color, lw=2, label=f'{cls}  (AP={ap:.3f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall'); ax.legend(loc='lower left'); ax.grid(alpha=0.3)

fig.suptitle('ROC / PR — TEST (K-fold + TTA)', fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/coffee_outputs/fig_roc_pr_v2.png', dpi=160, bbox_inches='tight')
plt.show()


In [ ]:
# Calibration  ─────────────────────────────────────────
from sklearn.calibration import calibration_curve

y_prob_max = y_proba_test.max(axis=1)
y_correct  = (y_pred_test == y_true_test).astype(int)
fop, mpv = calibration_curve(y_correct, y_prob_max, n_bins=10, strategy='uniform')


def expected_calibration_error(probs, correct, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1); e = 0.0
    for i in range(n_bins):
        m = (probs >= edges[i]) & (probs < edges[i+1])
        if m.sum() > 0:
            e += (m.sum() / len(probs)) * abs(correct[m].mean() - probs[m].mean())
    return e


ece_v = expected_calibration_error(y_prob_max, y_correct)

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect')
ax.plot(mpv, fop, 'o-', color='#d62728', lw=2, ms=8, label='Model')
ax.fill_between(mpv, mpv, fop, alpha=0.2, color='#d62728')
ax.set_xlabel('Mean predicted confidence'); ax.set_ylabel('Empirical accuracy')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title(f'Reliability  (ECE = {ece_v:.4f})', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/coffee_outputs/fig_calibration_v2.png', dpi=160, bbox_inches='tight')
plt.show()


## 11.1 Temperature scaling — a cheap calibration improvement

After ensemble + TTA, softmax is usually over-confident. Fit `T` (1 scalar) on the val set via NLL minimization, then apply to test → ECE drops while accuracy is unchanged.


In [ ]:
# Temperature scaling on val set, apply to test  ────
if RUN_TEMPERATURE_SCALING:
    print('Re-extracting K-fold ensemble probs on VAL for T-scaling...')
    ft_val_ds = CoffeeDatasetFT(val_samples, val_transform, le)
    ft_val_loader = DataLoader(
        ft_val_ds, batch_size=FT_BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'),
        worker_init_fn=seed_worker, generator=GENERATOR,
    )
    backbone_v = AutoModel.from_pretrained(DINOV2_MODEL)
    model_v = DINOv2FineTuner(backbone_v, n_classes=4, n_unfreeze=3, p_drop=0.3).to(DEVICE)

    @torch.no_grad()
    def _logits_from(model, loader):
        model.eval(); all_l, all_y = [], []
        use_amp = (DEVICE.type == 'cuda')
        for imgs, labs in loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=use_amp):
                logits, _ = model(imgs)
            all_l.append(logits.cpu().float().numpy())
            all_y.extend(labs.numpy())
        return np.concatenate(all_l, axis=0), np.array(all_y)

    val_logits = []
    for fi, state in enumerate(fold_states):
        model_v.load_state_dict(state)
        l, yv = _logits_from(model_v, ft_val_loader)
        val_logits.append(l)
    val_logits = np.mean(val_logits, axis=0)
    val_y = yv  # last (same across folds since loader is fixed)

    del model_v, backbone_v
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

    # Fit T by minimizing NLL on val (1-D scalar search)
    val_logits_t = torch.tensor(val_logits, dtype=torch.float32)
    val_y_t      = torch.tensor(val_y, dtype=torch.long)

    def nll_at(T):
        return F.cross_entropy(val_logits_t / T, val_y_t).item()

    # Coarse + fine search
    grid = np.concatenate([np.linspace(0.5, 1.0, 11), np.linspace(1.05, 4.0, 60)])
    losses = [nll_at(t) for t in grid]
    T_best = float(grid[int(np.argmin(losses))])
    print(f'Best T = {T_best:.3f}  (val NLL: '
          f'{nll_at(1.0):.4f} -> {nll_at(T_best):.4f})')

    # Apply to test logits (recover from softmax: log + arbitrary const)
    test_logits = np.log(np.clip(y_proba_test, 1e-12, 1.0))
    test_probs_T = F.softmax(torch.tensor(test_logits) / T_best, dim=-1).numpy()
    test_pred_T = test_probs_T.argmax(axis=1)

    ece_before = expected_calibration_error(y_proba_test.max(axis=1),
                                            (y_pred_test == y_true_test).astype(int))
    ece_after  = expected_calibration_error(test_probs_T.max(axis=1),
                                            (test_pred_T == y_true_test).astype(int))
    acc_after  = accuracy_score(y_true_test, test_pred_T)
    f1_after   = f1_score(y_true_test, test_pred_T, average='macro')

    print(f'\nTest set after T-scaling (T={T_best:.3f}):')
    print(f'  Acc  : {acc_tta:.4f} -> {acc_after:.4f}')
    print(f'  F1   : {f1_tta:.4f} -> {f1_after:.4f}')
    print(f'  ECE  : {ece_before:.4f} -> {ece_after:.4f}   '
          f'({(ece_before - ece_after)/max(ece_before,1e-9)*100:+.1f}%)')

    # Reliability before/after
    fop_b, mpv_b = calibration_curve((y_pred_test == y_true_test).astype(int),
                                     y_proba_test.max(axis=1), n_bins=10)
    fop_a, mpv_a = calibration_curve((test_pred_T == y_true_test).astype(int),
                                     test_probs_T.max(axis=1), n_bins=10)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, (mpv, fop, ece, ttl) in zip(axes,
        [(mpv_b, fop_b, ece_before, f'Before  ECE={ece_before:.4f}'),
         (mpv_a, fop_a, ece_after,  f'After T={T_best:.2f}  ECE={ece_after:.4f}')]):
        ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
        ax.plot(mpv, fop, 'o-', color='#d62728', lw=2, ms=8)
        ax.fill_between(mpv, mpv, fop, alpha=0.2, color='#d62728')
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.set_xlabel('Mean predicted confidence'); ax.set_ylabel('Empirical accuracy')
        ax.set_title(ttl, fontweight='bold'); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/coffee_outputs/fig_calibration_tscale.png', dpi=160, bbox_inches='tight')
    plt.show()
else:
    T_best = 1.0
    ece_after = ece_v
    print('Temperature scaling skipped (RUN_TEMPERATURE_SCALING=False).')


In [ ]:
# Hardest misclassifications  ───────────────────────────
mis_idx = np.where(y_pred_test != y_true_test)[0]
cor_idx = np.where(y_pred_test == y_true_test)[0]

n_mis = min(8, len(mis_idx))
top_mis = mis_idx[np.argsort(-y_proba_test[mis_idx].max(axis=1))[:n_mis]] if n_mis else np.array([], dtype=int)
np.random.seed(SEED)
n_cor = min(4, len(cor_idx))
top_cor = np.random.choice(cor_idx, n_cor, replace=False) if n_cor else np.array([], dtype=int)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for ax, i in zip(axes.flat[:n_mis], top_mis):
    img = Image.open(test_samples[i][0]).convert('RGB').resize((224, 224))
    ax.imshow(img); ax.axis('off')
    t = TARGET_CLASSES[y_true_test[i]]
    p = TARGET_CLASSES[y_pred_test[i]]
    ax.set_title(f'T:{t} -> P:{p}\nconf={y_proba_test[i].max()*100:.1f}%',
                 color='#d62728', fontsize=10)
for ax, i in zip(axes.flat[8:8+n_cor], top_cor):
    img = Image.open(test_samples[i][0]).convert('RGB').resize((224, 224))
    ax.imshow(img); ax.axis('off')
    ax.set_title(f'OK {TARGET_CLASSES[y_true_test[i]]}  '
                 f'conf={y_proba_test[i].max()*100:.1f}%',
                 color='#2ca02c', fontsize=10)
for ax in axes.flat[n_mis:8]:
    ax.axis('off')
for ax in axes.flat[8+n_cor:]:
    ax.axis('off')

fig.suptitle('Hardest misclassifications (top, red) & correct (bottom, green) — K-fold + TTA',
             fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/coffee_outputs/fig_misclass_v2.png', dpi=160, bbox_inches='tight')
plt.show()


## 12. Ablation studies

This is the part reviewers ask about most: which improvement actually contributes how much? The ablations run independently of the main pipeline, using the same 945-image test set for comparison:

| # | Ablation | Compute | Compared against? |
|---|---|---|---|
| 12.1 | **No-HC** (α=1.0, deep-only) | ~1 min | HC contribution |
| 12.2 | **ViT-B/14** instead of ViT-L | ~5 min | Backbone size contribution |
| 12.3 | **Single-FT** instead of K-fold | ~25 min | Ensemble contribution |
| 12.4 | Summary (table + delta) | ~5 sec | — |

> Turn off the `RUN_ABLATION_*` flags in Cell 5 if you are short on time.


In [ ]:
# 12.1 No-HC ablation: alpha = 1.0 (deep-only) ─────────
ablation_results = {}
ablation_preds   = {}

if RUN_ABLATION_NO_HC:
    print('=' * 60)
    print('  ABLATION 12.1 — No-HC (alpha = 1.0)')
    print('=' * 60)
    # Re-fuse with alpha=1.0 (deep-only) then re-run PCA + ELM ensemble
    ALPHA_NOHC = 1.0
    Xr_tr = np.concatenate([ALPHA_NOHC * X_train_deep_n,
                            (1 - ALPHA_NOHC) * X_train_hc_n], axis=1)
    Xr_va = np.concatenate([ALPHA_NOHC * X_val_deep_n,
                            (1 - ALPHA_NOHC) * X_val_hc_n], axis=1)
    Xr_te = np.concatenate([ALPHA_NOHC * X_test_deep_n,
                            (1 - ALPHA_NOHC) * X_test_hc_n], axis=1)
    pca_n = PCA(n_components=PCA_DIM, random_state=SEED, svd_solver='full').fit(Xr_tr)
    Xn_tr = pca_n.transform(Xr_tr); Xn_te = pca_n.transform(Xr_te)
    elms_n = []
    for i in range(N_ENSEMBLES):
        e = ELM(n_hidden=best_H, C=best_C, random_state=SEED + i)
        e.fit(Xn_tr, y_train_enc)
        elms_n.append(e)
    yp_nohc = ensemble_predict(Xn_te, elms_n, le)
    a, alo, ahi = bootstrap_metric(np.asarray(y_true), yp_nohc, acc_fn)
    f, flo, fhi = bootstrap_metric(np.asarray(y_true), yp_nohc, f1_fn)
    ablation_results['No-HC (alpha=1.0)'] = dict(
        acc=a, acc_lo=alo, acc_hi=ahi, f1=f, f1_lo=flo, f1_hi=fhi)
    ablation_preds  ['No-HC (alpha=1.0)'] = yp_nohc

    # McNemar vs HC=on (alpha=0.95)
    n01, n10, pv = mcnemar_exact(np.asarray(y_true),
                                 np.asarray(yp_nohc),
                                 np.asarray(y_pred_frozen))
    sig = 'SIG (p<0.05)' if pv < 0.05 else 'n.s.'
    print(f'No-HC : Acc={a:.4f} [{alo:.4f},{ahi:.4f}]  F1={f:.4f}')
    print(f'HC=on : Acc={acc_frozen:.4f}  F1={f1_frozen:.4f}')
    print(f'McNemar No-HC vs HC=on  : p={pv:.4f} [{sig}]  '
          f'(no-HC wins {n01} samples / loses {n10})')
else:
    print('Skip 12.1 (RUN_ABLATION_NO_HC=False).')


In [ ]:
# 12.2 ViT-B/14 ablation: same frozen + ELM pipeline, only the backbone is swapped
if RUN_ABLATION_VITB:
    print('=' * 60)
    print('  ABLATION 12.2 — ViT-B/14 (768d) instead of ViT-L/14 (1024d)')
    print('=' * 60)
    print(f'Loading {DINOV2_BASE}...')
    proc_b = AutoImageProcessor.from_pretrained(DINOV2_BASE)
    bb_b   = AutoModel.from_pretrained(DINOV2_BASE).to(DEVICE).eval()
    for p in bb_b.parameters(): p.requires_grad = False

    def collate_b(batch):
        imgs, lbls, paths = zip(*batch)
        ins = proc_b(images=list(imgs), return_tensors='pt',
                     size={'height': IMG_SIZE, 'width': IMG_SIZE})
        return ins, list(lbls), list(paths)

    @torch.no_grad()
    def extract_b(samples, desc):
        ds = CoffeeDataset(samples)
        loader = DataLoader(ds, batch_size=EXTRACT_BATCH, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'),
                            collate_fn=collate_b,
                            worker_init_fn=seed_worker, generator=GENERATOR)
        feats, labels = [], []
        use_amp = (DEVICE.type == 'cuda')
        for ins, lbls, _ in tqdm(loader, desc=desc):
            px = ins['pixel_values'].to(DEVICE, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=use_amp):
                out = bb_b(pixel_values=px)
                cls = out.last_hidden_state[:, 0, :]
            feats.append(cls.cpu().float().numpy())
            labels.extend(lbls)
        return np.concatenate(feats, axis=0), labels

    Xb_tr, _ = extract_b(train_samples, 'B-Train')
    Xb_te, _ = extract_b(test_samples,  'B-Test ')
    Xb_tr = normalize(Xb_tr, norm='l2'); Xb_te = normalize(Xb_te, norm='l2')

    # Same fusion alpha + PCA pipeline (HC same as before)
    Xr_tr = np.concatenate([ALPHA * Xb_tr, (1 - ALPHA) * X_train_hc_n], axis=1)
    Xr_te = np.concatenate([ALPHA * Xb_te, (1 - ALPHA) * X_test_hc_n], axis=1)
    pca_b = PCA(n_components=min(PCA_DIM, Xr_tr.shape[1]),
                random_state=SEED, svd_solver='full').fit(Xr_tr)
    Xn_tr = pca_b.transform(Xr_tr); Xn_te = pca_b.transform(Xr_te)
    elms_b = []
    for i in range(N_ENSEMBLES):
        e = ELM(n_hidden=best_H, C=best_C, random_state=SEED + i)
        e.fit(Xn_tr, y_train_enc)
        elms_b.append(e)
    yp_b = ensemble_predict(Xn_te, elms_b, le)
    a, alo, ahi = bootstrap_metric(np.asarray(y_true), yp_b, acc_fn)
    f, flo, fhi = bootstrap_metric(np.asarray(y_true), yp_b, f1_fn)
    ablation_results['ViT-B Frozen + Ens ELM'] = dict(
        acc=a, acc_lo=alo, acc_hi=ahi, f1=f, f1_lo=flo, f1_hi=fhi)
    ablation_preds  ['ViT-B Frozen + Ens ELM'] = yp_b

    n01, n10, pv = mcnemar_exact(np.asarray(y_true),
                                 np.asarray(yp_b),
                                 np.asarray(y_pred_frozen))
    sig = 'SIG (p<0.05)' if pv < 0.05 else 'n.s.'
    print(f'ViT-B : Acc={a:.4f} [{alo:.4f},{ahi:.4f}]  F1={f:.4f}')
    print(f'ViT-L : Acc={acc_frozen:.4f}  F1={f1_frozen:.4f}')
    print(f'McNemar ViT-B vs ViT-L (frozen) : p={pv:.4f} [{sig}]')

    del bb_b, proc_b
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
else:
    print('Skip 12.2 (RUN_ABLATION_VITB=False).')


In [ ]:
# 12.3 Single-FT ablation: 1 FT model on (train+val) with internal 85/15 split
if RUN_ABLATION_SINGLE_FT:
    print('=' * 60)
    print('  ABLATION 12.3 — Single FT (no K-fold)')
    print('=' * 60)
    p_tr2, p_va2, l_tr2, l_va2 = train_test_split(
        tv_paths, tv_labels, test_size=0.15,
        stratify=tv_labels, random_state=SEED,
    )
    s_train = list(zip(p_tr2, l_tr2))
    s_val   = list(zip(p_va2, l_va2))

    state_s, val_f1_s = train_one_model(s_train, s_val,
                                        tag='single-FT', epochs=FT_EPOCHS)

    backbone_s = AutoModel.from_pretrained(DINOV2_MODEL)
    model_s = DINOv2FineTuner(backbone_s, n_classes=4, n_unfreeze=3, p_drop=0.3).to(DEVICE)
    model_s.load_state_dict(state_s)

    # Plain test (no TTA) prob to keep apples-to-apples vs K-fold no-TTA
    test_probs_s, _ = softmax_predict(model_s, ft_test_loader)
    yp_s = test_probs_s.argmax(axis=1)

    a, alo, ahi = bootstrap_metric(np.asarray(y_true_test), yp_s, acc_fn)
    f, flo, fhi = bootstrap_metric(np.asarray(y_true_test), yp_s, f1_fn)
    ablation_results['Single FT (no TTA)'] = dict(
        acc=a, acc_lo=alo, acc_hi=ahi, f1=f, f1_lo=flo, f1_hi=fhi)
    ablation_preds  ['Single FT (no TTA)'] = le.inverse_transform(yp_s)

    n01, n10, pv = mcnemar_exact(np.asarray(y_true_test),
                                 np.asarray(yp_s),
                                 np.asarray(y_pred_kfold))
    sig = 'SIG (p<0.05)' if pv < 0.05 else 'n.s.'
    print(f'Single FT  : Acc={a:.4f} [{alo:.4f},{ahi:.4f}]  F1={f:.4f}  '
          f'(val F1={val_f1_s:.4f})')
    print(f'K-fold     : Acc={acc_kfold:.4f}  F1={f1_kfold:.4f}')
    print(f'McNemar Single-FT vs K-fold : p={pv:.4f} [{sig}]')

    del model_s, backbone_s
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
else:
    print('Skip 12.3 (RUN_ABLATION_SINGLE_FT=False).')


In [ ]:
# 12.4 Ablation summary table  ─────────────────────────
all_rows = []

# Baseline rows from main pipeline
for nm, key in [('Frozen + Ens ELM (HC=on, alpha=0.95)', 'Frozen + Ens ELM'),
                ('K-fold (no TTA)',                       'K-fold (no TTA)'),
                ('K-fold + TTA *',                        'K-fold + TTA *')]:
    if key in ci_table:
        r = ci_table[key]
        all_rows.append(dict(method=nm,
                             acc=r['acc'], acc_lo=r['acc_lo'], acc_hi=r['acc_hi'],
                             f1=r['f1'],   f1_lo=r['f1_lo'],   f1_hi=r['f1_hi']))

# Ablation rows
for nm, r in ablation_results.items():
    all_rows.append(dict(method=nm, **r))

print('=' * 100)
print(f'  {"ABLATION SUMMARY (95% CI, " + str(N_BOOTSTRAP) + " bootstrap)":^96}')
print('=' * 100)
print(f'  {"Method":<46} {"Acc [95% CI]":<25} {"F1 [95% CI]":<25}')
print('-' * 100)
for r in all_rows:
    a_s = f'{r["acc"]:.4f}[{r["acc_lo"]:.4f},{r["acc_hi"]:.4f}]'
    f_s = f'{r["f1"]:.4f}[{r["f1_lo"]:.4f},{r["f1_hi"]:.4f}]'
    print(f'  {r["method"]:<46} {a_s:<25} {f_s:<25}')
print('=' * 100)

# Delta contribution analysis (if all ablations ran)
print('\n--- Delta contribution analysis ---')
def _get(name):
    for r in all_rows:
        if r['method'] == name: return r
    return None

base    = _get('Frozen + Ens ELM (HC=on, alpha=0.95)')
no_hc   = _get('No-HC (alpha=1.0)')
vitb    = _get('ViT-B Frozen + Ens ELM')
single  = _get('Single FT (no TTA)')
kfold   = _get('K-fold (no TTA)')
kf_tta  = _get('K-fold + TTA *')

if base and no_hc:
    print(f'HC contribution    (frozen):  Acc {(base["acc"]-no_hc["acc"])*100:+.2f}pp  '
          f'F1 {(base["f1"]-no_hc["f1"])*100:+.2f}pp')
if base and vitb:
    print(f'ViT-L vs ViT-B     (frozen):  Acc {(base["acc"]-vitb["acc"])*100:+.2f}pp  '
          f'F1 {(base["f1"]-vitb["f1"])*100:+.2f}pp')
if single and kfold:
    print(f'K-fold vs Single-FT       :  Acc {(kfold["acc"]-single["acc"])*100:+.2f}pp  '
          f'F1 {(kfold["f1"]-single["f1"])*100:+.2f}pp')
if kfold and kf_tta:
    print(f'TTA contribution          :  Acc {(kf_tta["acc"]-kfold["acc"])*100:+.2f}pp  '
          f'F1 {(kf_tta["f1"]-kfold["f1"])*100:+.2f}pp')

pd.DataFrame(all_rows).to_csv('/content/drive/MyDrive/coffee_outputs/ablation_summary_v2_1.csv', index=False)
print('\n[OK] ablation_summary_v2_1.csv saved')


## 13. Summary & **Save all models into per-group folders**

Save all trained models (SVM, ELM, Ens-ELM, RF, XGBoost, K-fold FT, ablations) into a separate folder for each group, along with a `meta.json` holding hyper-params + test metrics + an overall `config.json` + a `README.md` with reload instructions.


In [ ]:
print('=' * 70)
print('   PIPELINE v2 — SUMMARY')
print('=' * 70)
print(f'  Train: {len(train_samples):,}  Val: {len(val_samples):,}  Test: {len(test_samples):,}')
print(f'  Classes  : {TARGET_CLASSES}')
print('=' * 70)
print(f'  Backbone : DINOv2 ViT-L/14 ({DEEP_DIM}-dim CLS)')
print(f'  HC       : 136-dim (HSV 96 + Gabor 40)')
print(f'  Fusion α : {ALPHA}  |  PCA {PCA_DIM}-dim')
print(f'  K-fold   : {N_FOLDS} folds, {FT_EPOCHS} epochs/fold')
print(f'  TTA      : {TTA_SCALES} x 4 flips = 12 passes')
print('=' * 70)
print('  RESULTS:')
print(f'    Frozen + Ens ELM     : Acc={acc_frozen:.4f}  F1={f1_frozen:.4f}')
print(f'    K-fold (no TTA)      : Acc={acc_kfold:.4f}  F1={f1_kfold:.4f}')
print(f'    K-fold + multi-TTA * : Acc={acc_tta:.4f}  F1={f1_tta:.4f}')
print('=' * 70)


In [ ]:
# SAVE ALL MODELS INTO PER-GROUP FOLDERS  ──────────────────
# Folder structure:
#   /content/drive/MyDrive/coffee_outputs/models/
#     01_svm/                       SVM (rbf, C=10)
#     02_elm_single/                ELM don (best from grid search)
#     03_ensemble_elm/              Ensemble ELM 10x (deep+HC+PCA)
#     04_random_forest/             RandomForest 300 trees
#     05_xgboost/                   XGBoost
#     06_kfold_finetune/            5-fold fine-tuned DINOv2 ViT-L
#     07_ablation_vitb/             ViT-B ablation (Ens ELM)
#     08_ablation_nohc/             No-HC ablation (alpha=1.0, Ens ELM)
#     09_ablation_single_ft/        Single FT (no K-fold) ablation
#     preprocessing/                PCA, label encoder
#     config.json                   Overall config and metrics
#     README.md                     Instructions to reload/reuse models

import joblib, json
import shutil
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/coffee_outputs/models')
if ROOT.exists():
    shutil.rmtree(ROOT)                          # reset old folder
ROOT.mkdir(parents=True, exist_ok=True)


def _save_elm_list(elms, folder):
    """Serialize one ELM list into a list of dicts (keeping numpy arrays)."""
    arts = []
    for e in elms:
        arts.append({
            'n_hidden': e.n_hidden, 'activation': e.activation,
            'C': e.C, 'random_state': e.random_state,
            'normalize_H': e.normalize_H,
            'W': e.W, 'b': e.b, 'beta': e.beta,
            'classes_': e.classes_,
        })
    joblib.dump(arts, folder / 'elms.joblib', compress=3)
    return len(arts)


saved_log = {}

# 1. SVM
d1 = ROOT / '01_svm'; d1.mkdir()
joblib.dump(svm1, d1 / 'svm.joblib', compress=3)
with open(d1 / 'meta.json', 'w', encoding='utf-8') as f:
    json.dump({
        'model_type': 'sklearn.svm.SVC',
        'kernel': 'rbf', 'C': 10, 'probability': True,
        'input_features': 'PCA(deep ViT-L cls + HC, alpha=0.95)',
        'input_dim': int(X_train.shape[1]),
        'accuracy_test': float(compare['ViT-L Frozen+HC+PCA + SVM']['acc']),
        'f1_macro_test': float(compare['ViT-L Frozen+HC+PCA + SVM']['f1']),
    }, f, ensure_ascii=False, indent=2)
saved_log['01_svm'] = 'svm.joblib + meta.json'

# 2. ELM single
d2 = ROOT / '02_elm_single'; d2.mkdir()
joblib.dump({
    'n_hidden': elm_single.n_hidden, 'activation': elm_single.activation,
    'C': elm_single.C, 'random_state': elm_single.random_state,
    'normalize_H': elm_single.normalize_H,
    'W': elm_single.W, 'b': elm_single.b, 'beta': elm_single.beta,
    'classes_': elm_single.classes_,
}, d2 / 'elm.joblib', compress=3)
with open(d2 / 'meta.json', 'w', encoding='utf-8') as f:
    json.dump({
        'model_type': 'ELM (custom, ridge + L2-norm hidden)',
        'n_hidden': best_H, 'C': best_C, 'activation': 'relu',
        'best_cv_f1': float(best_f1),
        'input_dim': int(X_train.shape[1]),
        'accuracy_test': float(compare['ViT-L Frozen+HC+PCA + ELM']['acc']),
        'f1_macro_test': float(compare['ViT-L Frozen+HC+PCA + ELM']['f1']),
    }, f, ensure_ascii=False, indent=2)
saved_log['02_elm_single'] = 'elm.joblib + meta.json'

# 3. Ensemble ELM (10x)
d3 = ROOT / '03_ensemble_elm'; d3.mkdir()
n3 = _save_elm_list(ensemble_elms, d3)
with open(d3 / 'meta.json', 'w', encoding='utf-8') as f:
    json.dump({
        'model_type': 'Ensemble ELM (soft voting)',
        'n_ensembles': n3,
        'n_hidden': best_H, 'C': best_C, 'activation': 'relu',
        'seeds': [SEED + i for i in range(n3)],
        'input_dim': int(X_train.shape[1]),
        'accuracy_test': float(acc_frozen),
        'f1_macro_test': float(f1_frozen),
    }, f, ensure_ascii=False, indent=2)
saved_log['03_ensemble_elm'] = f'elms.joblib ({n3} models) + meta.json'

# 4. Random Forest
d4 = ROOT / '04_random_forest'; d4.mkdir()
joblib.dump(rf, d4 / 'rf.joblib', compress=3)
with open(d4 / 'meta.json', 'w', encoding='utf-8') as f:
    json.dump({
        'model_type': 'sklearn.ensemble.RandomForestClassifier',
        'n_estimators': 300, 'random_state': SEED,
        'input_dim': int(X_train.shape[1]),
        'accuracy_test': float(compare['ViT-L Frozen+HC+PCA + Random Forest']['acc']),
        'f1_macro_test': float(compare['ViT-L Frozen+HC+PCA + Random Forest']['f1']),
    }, f, ensure_ascii=False, indent=2)
saved_log['04_random_forest'] = 'rf.joblib + meta.json'

# 5. XGBoost
d5 = ROOT / '05_xgboost'; d5.mkdir()
xgb_clf.save_model(str(d5 / 'xgb.json'))         # native XGB format
joblib.dump(xgb_clf, d5 / 'xgb_clf.joblib', compress=3)
with open(d5 / 'meta.json', 'w', encoding='utf-8') as f:
    json.dump({
        'model_type': 'xgboost.XGBClassifier',
        'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05,
        'subsample': 0.8, 'colsample_bytree': 0.8, 'eval_metric': 'mlogloss',
        'tree_method': 'hist', 'random_state': SEED,
        'input_dim': int(X_train.shape[1]),
        'accuracy_test': float(compare['ViT-L Frozen+HC+PCA + XGBoost']['acc']),
        'f1_macro_test': float(compare['ViT-L Frozen+HC+PCA + XGBoost']['f1']),
    }, f, ensure_ascii=False, indent=2)
saved_log['05_xgboost'] = 'xgb.json + xgb_clf.joblib + meta.json'

# 6. K-fold fine-tune (5 fold weights)
d6 = ROOT / '06_kfold_finetune'; d6.mkdir()
for fi, state in enumerate(fold_states):
    torch.save(state, d6 / f'ft_fold{fi+1}.pth')
with open(d6 / 'meta.json', 'w', encoding='utf-8') as f:
    json.dump({
        'model_type': 'DINOv2FineTuner (ViT-L/14 + 3 unfrozen blocks + LN + custom head)',
        'backbone': DINOV2_MODEL,
        'deep_dim': DEEP_DIM,
        'n_classes': 4,
        'n_unfreeze_blocks': 3,
        'dropout': 0.3,
        'head_arch': 'LN -> Dropout(0.3) -> Linear(1024->256) -> GELU -> Dropout(0.225) -> Linear(256->4)',
        'n_folds': N_FOLDS,
        'epochs_per_fold': FT_EPOCHS,
        'patience': PATIENCE,
        'img_size': IMG_SIZE,
        'batch_size': FT_BATCH_SIZE,
        'optimizer': 'AdamW (backbone lr=1e-5 wd=5e-4, head lr=1e-4 wd=5e-2)',
        'scheduler': 'CosineAnnealingLR(eta_min=1e-6)',
        'tricks': ['mixup(0.2)+cutmix(1.0) p=0.5/0.5', 'label_smoothing=0.1',
                   'EMA(0.999)', 'grad_clip=1.0', 'AMP fp16', 'WeightedRandomSampler'],
        'fold_dev_f1': [float(x) for x in fold_dev_f1],
        'accuracy_test_kfold_no_tta': float(acc_kfold),
        'f1_macro_test_kfold_no_tta': float(f1_kfold),
        'accuracy_test_kfold_tta': float(acc_tta),
        'f1_macro_test_kfold_tta': float(f1_tta),
        'tta_scales': TTA_SCALES,
    }, f, ensure_ascii=False, indent=2)
saved_log['06_kfold_finetune'] = f'{N_FOLDS} x ft_foldX.pth + meta.json'

# 7. Ablation ViT-B (Frozen + Ens ELM)
if RUN_ABLATION_VITB and 'elms_b' in globals():
    d7 = ROOT / '07_ablation_vitb'; d7.mkdir()
    n7 = _save_elm_list(elms_b, d7)
    joblib.dump(pca_b, d7 / 'pca.joblib', compress=3)
    abl_b = ablation_results.get('ViT-B Frozen + Ens ELM', {})
    with open(d7 / 'meta.json', 'w', encoding='utf-8') as f:
        json.dump({
            'model_type': 'Ensemble ELM on DINOv2 ViT-B/14 frozen features (ablation)',
            'backbone': DINOV2_BASE, 'deep_dim': DEEP_DIM_B,
            'n_ensembles': n7,
            'n_hidden': best_H, 'C': best_C,
            'accuracy_test': float(abl_b.get('acc', float('nan'))),
            'f1_macro_test': float(abl_b.get('f1', float('nan'))),
        }, f, ensure_ascii=False, indent=2)
    saved_log['07_ablation_vitb'] = f'elms.joblib ({n7}) + pca.joblib + meta.json'
else:
    print('  [skip] 07_ablation_vitb (RUN_ABLATION_VITB=False or not trained)')

# 8. Ablation No-HC (alpha=1.0)
if RUN_ABLATION_NO_HC and 'elms_n' in globals():
    d8 = ROOT / '08_ablation_nohc'; d8.mkdir()
    n8 = _save_elm_list(elms_n, d8)
    joblib.dump(pca_n, d8 / 'pca.joblib', compress=3)
    abl_n = ablation_results.get('No-HC (alpha=1.0)', {})
    with open(d8 / 'meta.json', 'w', encoding='utf-8') as f:
        json.dump({
            'model_type': 'Ensemble ELM no-HC (alpha=1.0, deep-only)',
            'backbone': DINOV2_MODEL, 'alpha': 1.0,
            'n_ensembles': n8,
            'n_hidden': best_H, 'C': best_C,
            'accuracy_test': float(abl_n.get('acc', float('nan'))),
            'f1_macro_test': float(abl_n.get('f1', float('nan'))),
        }, f, ensure_ascii=False, indent=2)
    saved_log['08_ablation_nohc'] = f'elms.joblib ({n8}) + pca.joblib + meta.json'
else:
    print('  [skip] 08_ablation_nohc (RUN_ABLATION_NO_HC=False or not trained)')

# 9. Ablation Single FT (no K-fold)
if RUN_ABLATION_SINGLE_FT and 'state_s' in globals():
    d9 = ROOT / '09_ablation_single_ft'; d9.mkdir()
    torch.save(state_s, d9 / 'single_ft.pth')
    abl_s = ablation_results.get('Single FT (no TTA)', {})
    with open(d9 / 'meta.json', 'w', encoding='utf-8') as f:
        json.dump({
            'model_type': 'DINOv2FineTuner (single, no K-fold) — ablation',
            'backbone': DINOV2_MODEL, 'deep_dim': DEEP_DIM,
            'train_val_split': '85/15 inside (train+val)',
            'accuracy_test': float(abl_s.get('acc', float('nan'))),
            'f1_macro_test': float(abl_s.get('f1', float('nan'))),
        }, f, ensure_ascii=False, indent=2)
    saved_log['09_ablation_single_ft'] = 'single_ft.pth + meta.json'
else:
    print('  [skip] 09_ablation_single_ft (RUN_ABLATION_SINGLE_FT=False or not trained)')

# 10. Preprocessing (PCA + LabelEncoder)
dp = ROOT / 'preprocessing'; dp.mkdir()
joblib.dump(pca, dp / 'pca.joblib', compress=3)
joblib.dump(le, dp / 'label_encoder.joblib')
with open(dp / 'meta.json', 'w', encoding='utf-8') as f:
    json.dump({
        'pca_input_dim': int(DEEP_DIM + X_train_hc.shape[1]),
        'pca_output_dim': int(PCA_DIM),
        'var_kept': float(VAR_THRESH),
        'fusion_alpha': ALPHA,
        'l2_normalize': True,
        'hc_dim': int(X_train_hc.shape[1]),
        'deep_dim': DEEP_DIM,
        'label_encoder_classes': list(le.classes_),
    }, f, ensure_ascii=False, indent=2)
saved_log['preprocessing'] = 'pca.joblib + label_encoder.joblib + meta.json'

# Overall config (1 file to reload the full pipeline)
config = {
    'pipeline_version': 'v2.1',
    'backbone'       : DINOV2_MODEL,
    'deep_dim'       : DEEP_DIM,
    'hc_dim'         : int(X_train_hc.shape[1]),
    'pca_dim'        : int(PCA_DIM),
    'alpha'          : ALPHA,
    'n_folds'        : N_FOLDS,
    'ft_epochs'      : FT_EPOCHS,
    'tta_scales'     : TTA_SCALES,
    'img_size'       : IMG_SIZE,
    'classes'        : list(le.classes_),
    'classes_vi'     : CLASS_VI,
    'metrics': {
        'frozen_ensemble_elm': {'acc': float(acc_frozen),    'f1': float(f1_frozen)},
        'kfold_no_tta'      : {'acc': float(acc_kfold),     'f1': float(f1_kfold)},
        'kfold_tta'         : {'acc': float(acc_tta),       'f1': float(f1_tta)},
    },
    'fold_dev_f1'    : [float(x) for x in fold_dev_f1],
    'best_elm_hparams': {'n_hidden': int(best_H), 'C': float(best_C)},
    'n_test_samples' : int(N_TEST),
}
with open(ROOT / 'config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

# README for later reuse
readme = (
    "# Coffee Leaf Disease - DINOv2 ViT-L Pipeline v2.1 (saved models)\n\n"
    "## Structure\n"
    "Each folder holds one model group + meta.json (hyper-params + test metrics).\n\n"
    "| Folder | Description |\n"
    "|---|---|\n"
    "| 01_svm                 | SVM rbf C=10 on PCA features |\n"
    "| 02_elm_single          | 1 ELM (best_H, best_C from grid search) |\n"
    "| 03_ensemble_elm        | 10 ELM soft-voting (BASELINE frozen path) |\n"
    "| 04_random_forest       | RandomForest 300 trees |\n"
    "| 05_xgboost             | XGBoost 300 trees, max_depth=6 |\n"
    "| 06_kfold_finetune      | 5 fold DINOv2 ViT-L fine-tuned (BEST) |\n"
    "| 07_ablation_vitb       | Ablation: ViT-B instead of ViT-L |\n"
    "| 08_ablation_nohc       | Ablation: alpha=1.0 (no handcrafted) |\n"
    "| 09_ablation_single_ft  | Ablation: 1 FT model, no K-fold |\n"
    "| preprocessing          | PCA + LabelEncoder (shared by the frozen path) |\n"
    "| config.json            | Overall config + metrics |\n\n"
    "## Reload a model\n"
    "```python\n"
    "import joblib, torch, json\n"
    "from pathlib import Path\n"
    "R = Path('models')\n\n"
    "# (a) Frozen path: load preprocessing + ensemble ELM\n"
    "pca = joblib.load(R / 'preprocessing/pca.joblib')\n"
    "le  = joblib.load(R / 'preprocessing/label_encoder.joblib')\n"
    "elm_arts = joblib.load(R / '03_ensemble_elm/elms.joblib')\n"
    "# Recreate the ELM from the artifact (see the ELM class in the notebook)\n\n"
    "# (b) K-fold FT: load 5 state_dict\n"
    "fold_states = [torch.load(R / f'06_kfold_finetune/ft_fold{i+1}.pth') for i in range(5)]\n"
    "```\n"
)
with open(ROOT / 'README.md', 'w', encoding='utf-8') as f:
    f.write(readme)
saved_log['root'] = 'config.json + README.md'


# ---- Print summary  --------------------------------------
print('\n' + '=' * 78)
print(f'  {"MODELS SAVED INTO PER-GROUP FOLDERS":^74}')
print('=' * 78)
print(f'  Root: {ROOT}\n')
total_size = 0
for folder in sorted(ROOT.iterdir()):
    if folder.is_dir():
        size_b = sum(f.stat().st_size for f in folder.rglob('*') if f.is_file())
        n_files = sum(1 for _ in folder.rglob('*') if _.is_file())
        total_size += size_b
        print(f'  {folder.name:<26}  {n_files:>2} file(s)  {size_b/1024/1024:>8.2f} MB')
    else:
        total_size += folder.stat().st_size
        print(f'  {folder.name:<26}            {folder.stat().st_size/1024:>8.1f} KB')
print('-' * 78)
print(f'  {"TOTAL SIZE":<26}            {total_size/1024/1024:>8.2f} MB')
print('=' * 78)

# Save catalog
with open(ROOT / 'catalog.json', 'w', encoding='utf-8') as f:
    json.dump(saved_log, f, ensure_ascii=False, indent=2)
print(f'\n[OK] Catalog saved -> {ROOT / "catalog.json"}')


In [ ]:
# LaTeX table (paste vao paper) — v2.1 with bootstrap CI  ─────
from sklearn.metrics import precision_recall_fscore_support
p_cls, r_cls, f_cls, sup = precision_recall_fscore_support(
    y_true_test, y_pred_test, zero_division=0,
)

# Per-class table
print('% --- Per-class results table (v2.1) ---')
print(r'\begin{table}[t]')
print(r'\centering')
print(r'\caption{Per-class results on stratified test split — DINOv2 ViT-L + K-fold + TTA.}')
print(r'\label{tab:per_class_v2_1}')
print(r'\begin{tabular}{lcccc}')
print(r'\toprule')
print(r'Class & Precision (\%) & Recall (\%) & F1 (\%) & Support \\')
print(r'\midrule')
for cls, pp, rr, ff, ss in zip(TARGET_CLASSES, p_cls, r_cls, f_cls, sup):
    print(f'{cls} & {pp*100:.2f} & {rr*100:.2f} & {ff*100:.2f} & {ss} \\\\')
print(r'\midrule')
print(f'Macro-avg & {p_cls.mean()*100:.2f} & {r_cls.mean()*100:.2f} & '
      f'{f_cls.mean()*100:.2f} & {sup.sum()} \\\\')
print(r'\midrule')
print(f'\\multicolumn{{4}}{{l}}{{Overall accuracy}} & {acc_tta*100:.2f}\\% \\\\')
print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\end{table}')

# Method comparison table with 95% CI
print('\n% --- Method comparison with 95% bootstrap CI (v2.1) ---')
print(r'\begin{table}[t]')
print(r'\centering')
print(r'\caption{Method comparison on stratified test split (n=' + str(len(y_true_test))
      + r') with 95\% bootstrap confidence intervals (' + str(N_BOOTSTRAP) + ' resamples).}')
print(r'\label{tab:methods_ci_v2_1}')
print(r'\begin{tabular}{lcc}')
print(r'\toprule')
print(r'Method & Accuracy (\%) [95\% CI] & F1-macro (\%) [95\% CI] \\')
print(r'\midrule')
for name, r in sorted_compare.items():
    a_s = (f'{r["acc"]*100:.2f} [{r.get("acc_lo", r["acc"])*100:.2f}, '
           f'{r.get("acc_hi", r["acc"])*100:.2f}]')
    f_s = (f'{r["f1"]*100:.2f} [{r.get("f1_lo", r["f1"])*100:.2f}, '
           f'{r.get("f1_hi", r["f1"])*100:.2f}]')
    print(f'{name} & {a_s} & {f_s} \\\\')
print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\end{table}')

# Save CSVs
pd.DataFrame({
    'class': TARGET_CLASSES,
    'precision': p_cls, 'recall': r_cls, 'f1': f_cls, 'support': sup,
}).to_csv('/content/drive/MyDrive/coffee_outputs/per_class_v2_1.csv', index=False)
print('\n[OK] per_class_v2_1.csv saved')
